## 1. 기본 패키지 불러오기
ML feature process에 필요한 패키지를 불러옵니다. 기존 처리 로직과 경로는 바꾸지 않았습니다.

In [ ]:
# 이 셀은 ML 산출물 생성에 필요한 패키지만 불러옵니다.
from pathlib import Path
import os
import warnings

import numpy as np
import pandas as pd
import polars as pl

warnings.filterwarnings("ignore")

print("polars:", pl.__version__)
print("pandas:", pd.__version__)
print("numpy:", np.__version__)


## 2. 팀 공통 Google Drive 경로 설정
02번 preprocessing에서 생성한 `clean_base.parquet`을 읽고, ML용 산출물을 저장할 위치를 지정합니다.

In [ ]:
from pathlib import Path

# ============================================================
# Colab + Google Drive / Local path setup
# ============================================================
# 팀원 공통 실행 기준:
# - raw 원본 데이터: 기업 Google Drive
# - 큰 parquet / 팀 공유 산출물: 기업 Google Drive
# - 가벼운 검증표 / 피처 명세서 / 요약 CSV: Colab local
#
# Drive 폴더 구조:
# GraphAML/
# └── data/
#     ├── code/
#     ├── raw/
#     ├── processed/
#     │   ├── step01_clean_base/
#     │   └── ml_features/
#     └── graph/
#         └── gnn_inputs/
#
# Local 폴더 구조:
# /content/GraphAML_local_outputs/
# ├── step01_clean_base/
# ├── ml_features/
# └── gnn_inputs/
# ============================================================

try:
    from google.colab import drive
    drive.mount("/content/drive")
except ModuleNotFoundError:
    raise RuntimeError(
        "이 노트북은 기업용 Colab + Google Drive 실행 기준입니다. "
        "Colab에서 열고 Google Drive를 마운트한 뒤 실행해주세요."
    )

# Drive: raw 데이터, 큰 parquet, 팀 공유 산출물
DRIVE_BASE_DIR = Path("/content/drive/MyDrive/GraphAML")
DRIVE_DATA_DIR = DRIVE_BASE_DIR / "data"

CODE_DIR = DRIVE_DATA_DIR / "code"
RAW_DIR = DRIVE_DATA_DIR / "raw"
PROCESSED_DIR = DRIVE_DATA_DIR / "processed"
GRAPH_DIR = DRIVE_DATA_DIR / "graph"

STEP01_DRIVE_DIR = PROCESSED_DIR / "step01_clean_base"
ML_DRIVE_DIR = PROCESSED_DIR / "ml_features"
GNN_DRIVE_DIR = GRAPH_DIR / "gnn_inputs"

# Local: 가벼운 검증표, 피처 명세서, 요약 CSV
# 주의: Colab local이므로 런타임 종료 시 사라질 수 있습니다.
LOCAL_BASE_DIR = Path("/content/GraphAML_local_outputs")
STEP01_LOCAL_DIR = LOCAL_BASE_DIR / "step01_clean_base"
ML_LOCAL_DIR = LOCAL_BASE_DIR / "ml_features"
GNN_LOCAL_DIR = LOCAL_BASE_DIR / "gnn_inputs"

# 기존 코드 호환용 alias
BASE_DIR = DRIVE_BASE_DIR
DATA_DIR = DRIVE_DATA_DIR
STEP01_DIR = STEP01_DRIVE_DIR
ML_DIR = ML_DRIVE_DIR
GNN_DIR = GNN_DRIVE_DIR
PROJECT_DIR = DRIVE_BASE_DIR

for p in [
    CODE_DIR,
    RAW_DIR,
    PROCESSED_DIR,
    GRAPH_DIR,
    STEP01_DRIVE_DIR,
    ML_DRIVE_DIR,
    GNN_DRIVE_DIR,
    STEP01_LOCAL_DIR,
    ML_LOCAL_DIR,
    GNN_LOCAL_DIR,
]:
    p.mkdir(parents=True, exist_ok=True)

print("DRIVE_BASE_DIR   :", DRIVE_BASE_DIR)
print("DRIVE_DATA_DIR   :", DRIVE_DATA_DIR)
print("RAW_DIR          :", RAW_DIR)
print("STEP01_DRIVE_DIR :", STEP01_DRIVE_DIR)
print("ML_DRIVE_DIR     :", ML_DRIVE_DIR)
print("GNN_DRIVE_DIR    :", GNN_DRIVE_DIR)
print("LOCAL_BASE_DIR   :", LOCAL_BASE_DIR)
print("STEP01_LOCAL_DIR :", STEP01_LOCAL_DIR)
print("ML_LOCAL_DIR     :", ML_LOCAL_DIR)
print("GNN_LOCAL_DIR    :", GNN_LOCAL_DIR)
CONFIG = {
    "BASE_DIR": BASE_DIR,
    "DATA_DIR": DATA_DIR,

    # 02번 산출물 위치 후보. 새 Drive 구조를 우선 사용합니다.
    "CLEAN_BASE_CANDIDATES": [
        STEP01_DIR / "clean_base.parquet",
        PROCESSED_DIR / "clean_base.parquet",
    ],

    # 큰 parquet 산출물은 Drive에 저장
    "DRIVE_OUTPUT_DIR": ML_DRIVE_DIR,

    # 가벼운 검증표 / 피처 명세서 / 요약 CSV는 Colab local에 저장
    "LOCAL_OUTPUT_DIR": ML_LOCAL_DIR,

    # 기존 코드 호환용: 03번의 주 산출물 위치는 Drive
    "OUTPUT_DIR": ML_DRIVE_DIR,

    "RANDOM_SEED": 42,

    # 직접 재계산 검증은 비용 큼. 기본은 샘플 검증
    # 최종 제출 전 VALIDATION_MODE="full" 확인
    "SAMPLE_MODE": True,
    # 개발 중에는 샘플 검증을 쓰고, 최종 공유 전에는 full로 바꾸면 됩니다.
    "VALIDATION_MODE": "sample",  # "sample" 또는 "full"
    "MAX_VALIDATION_ROWS": 5000,

    "TRAIN_RATIO": 0.60,
    "VAL_RATIO": 0.20,
    "TEST_RATIO": 0.20,

    "WINDOWS": {
        "w1h": "1h",
        "w6h": "6h",
        "w1d": "1d",
        "w3d": "3d",
        "w7d": "7d",
    },

    # Small 데이터에서는 30d를 기본 feature로 만들지 않음
    "MAKE_OPTIONAL_30D": False,

    "COLUMN_CANDIDATES": {
        "timestamp": [
            "timestamp", "Timestamp", "time", "Time", "datetime", "DateTime"
        ],
        "label": [
            "is_laundering", "Is_Laundering", "label", "target", "y"
        ],
        "amount": [
            "amount", "Amount", "amount_paid", "Amount Paid", "paid_amount"
        ],

        # 02번 표준 컬럼명을 우선한다.
        "sender_account": [
            "sender_account_id", "sender_account", "from_account", "Account", "account",
            "orig_account", "source_account", "From Account"
        ],
        "receiver_account": [
            "receiver_account_id", "receiver_account", "to_account", "Account.1",
            "beneficiary_account", "dest_account", "destination_account", "To Account"
        ],
        "from_bank": [
            "sender_bank_id", "from_bank", "From Bank", "sender_bank", "source_bank"
        ],
        "to_bank": [
            "receiver_bank_id", "to_bank", "To Bank", "receiver_bank", "destination_bank"
        ],

        "payment_currency": [
            "payment_currency", "Payment Currency", "currency", "Currency"
        ],
        "receiving_currency": [
            "receiving_currency", "Receiving Currency", "received_currency"
        ],

        "payment_format": [
            "payment_format", "Payment Format", "payment_type", "Payment Type", "format"
        ],
        "tx_id": [
            "tx_id", "transaction_id", "TransactionID", "transaction_id_raw"
        ],
    },

    # 모델 입력 feature 점검용.
    # exact forbidden은 컬럼명 전체 일치만 금지하고
    # substring forbidden은 라벨/그래프 누수성 토큰만 검사한다.
    "EXACT_FORBIDDEN_MODEL_COLUMNS": {
        "label", "is_laundering", "tx_id", "transaction_id",
        "split", "timestamp", "datetime",
    },
    "SUBSTRING_FORBIDDEN_MODEL_TOKENS": {
        "laundering", "attempt_id", "pattern",
        "pagerank", "centrality", "degree",
    },
}

CONFIG["DRIVE_OUTPUT_DIR"].mkdir(parents=True, exist_ok=True)
CONFIG["LOCAL_OUTPUT_DIR"].mkdir(parents=True, exist_ok=True)

print("BASE_DIR:", CONFIG["BASE_DIR"])
print("DATA_DIR:", CONFIG["DATA_DIR"])
print("DRIVE_OUT:", CONFIG["DRIVE_OUTPUT_DIR"])
print("LOCAL_OUT:", CONFIG["LOCAL_OUTPUT_DIR"])
print("clean_base candidates:")
for p in CONFIG["CLEAN_BASE_CANDIDATES"]:
    print(" -", p, "exists=", Path(p).exists())


## 3. 보조 함수 준비
split, feature catalog, leakage check 등에 반복해서 쓰는 함수입니다. 산출물의 신뢰도를 높이기 위한 검증 보조 코드입니다.

In [ ]:
# ============================================================
# 보조 함수
# ============================================================

MEMORY_RECORDS = []

# 후보 경로 중 실제 존재하는 파일을 선택합니다.
def resolve_existing_path(candidates, name="input"):
    for path in candidates:
        if Path(path).exists():
            print("{} path: {}".format(name, path))
            return Path(path)

    msg = "{} 파일을 찾지 못했습니다. 확인한 후보 경로:\n{}".format(
        name,
        "\n".join([str(p) for p in candidates])
    )
    raise FileNotFoundError(msg)


def get_memory_mb():
    try:
        import psutil
        process = psutil.Process(os.getpid())
        return process.memory_info().rss / 1024 / 1024
    except Exception:
        return np.nan


# 주요 단계별 메모리 사용량을 리포트용으로 기록합니다.
def record_memory(step, df=None):
    row_count = None
    col_count = None

    if df is not None:
        row_count = df.height
        col_count = len(df.columns)

    MEMORY_RECORDS.append({
        "step": step,
        "row_count": row_count,
        "col_count": col_count,
        "memory_mb": get_memory_mb(),
        "checked_at": pd.Timestamp.now().isoformat(),
    })

    print("[MEM]", step, "| rows=", row_count, "| cols=", col_count, "| mb=", round(get_memory_mb(), 2))


# 누적된 메모리 기록을 CSV로 저장합니다.
def save_memory_profile():
    path = CONFIG["LOCAL_OUTPUT_DIR"] / "memory_profile.csv"
    pd.DataFrame(MEMORY_RECORDS).to_csv(path, index=False, encoding="utf-8-sig")
    print("saved:", path)


def resolve_column(df, logical_name, required=True):
    candidates = CONFIG["COLUMN_CANDIDATES"].get(logical_name, [])
    existing = set(df.columns)

    for c in candidates:
        if c in existing:
            return c

    lower_map = {c.lower(): c for c in df.columns}
    for c in candidates:
        if c.lower() in lower_map:
            return lower_map[c.lower()]

    if required:
        raise ValueError(
            "필수 컬럼을 찾지 못했습니다. logical_name={}, candidates={}, columns={}".format(
                logical_name, candidates, df.columns
            )
        )

    return None


def safe_log1p_expr(col):
    return (
        pl.when(pl.col(col).is_null() | (pl.col(col) < 0))
        .then(None)
        .otherwise(pl.col(col).log1p())
    )


def parse_window_to_ns(window_str):
    unit = window_str[-1]
    value = int(window_str[:-1])

    if unit == "h":
        seconds = value * 3600
    elif unit == "d":
        seconds = value * 24 * 3600
    elif unit == "m":
        seconds = value * 60
    else:
        raise ValueError("지원하지 않는 window입니다: {}".format(window_str))

    return int(seconds * 1_000_000_000)


def series_datetime_to_ns(series):
    return series.to_numpy().astype("datetime64[ns]").astype("int64")


def safe_schema_get(schema, col, default="unknown"):
    return str(schema[col]) if col in schema else default


def print_shape(name, df):
    print("{}: rows={:,}, cols={:,}".format(name, df.height, len(df.columns)))


## 4. clean_base 로드
02번에서 만든 공통 기준 데이터를 읽습니다. 이 단계부터는 원본 CSV가 아니라 clean base를 기준으로 ML 파일을 만듭니다.

In [ ]:
# ============================================================
# clean_base 로드 및 기본 컬럼 정리
# ============================================================

# 01번 산출물 후보 중 실제 존재하는 clean_base를 읽습니다.
clean_path = resolve_existing_path(CONFIG["CLEAN_BASE_CANDIDATES"], name="clean_base")

df_raw = pl.read_parquet(clean_path)
record_memory("load_clean_base", df_raw)
print_shape("df_raw", df_raw)

COL = {
    "timestamp": resolve_column(df_raw, "timestamp", required=True),
    "label": resolve_column(df_raw, "label", required=True),
    "amount": resolve_column(df_raw, "amount", required=True),
    "sender_account": resolve_column(df_raw, "sender_account", required=True),
    "receiver_account": resolve_column(df_raw, "receiver_account", required=True),
    "from_bank": resolve_column(df_raw, "from_bank", required=False),
    "to_bank": resolve_column(df_raw, "to_bank", required=False),
    "payment_currency": resolve_column(df_raw, "payment_currency", required=False),
    "receiving_currency": resolve_column(df_raw, "receiving_currency", required=False),
    "payment_format": resolve_column(df_raw, "payment_format", required=False),
    "tx_id": resolve_column(df_raw, "tx_id", required=False),
}

print("매칭된 컬럼")
for k, v in COL.items():
    print("-", k, ":", v)

# clone 없이 바로 처리
df = df_raw

# 원본 clean_base에 없어서 unknown 상수로 만든 범주형 컬럼을 추적합니다.
# 이런 컬럼은 실제 원본 신호가 아니므로 인코딩 대상에서 제외합니다.
CREATED_UNKNOWN_CATEGORICAL_COLS = []

# tx_id 없으면 row 순서 기준 생성
if COL["tx_id"] is None:
    df = df.with_row_index("tx_id")
    COL["tx_id"] = "tx_id"

# 은행 / 통화 / 결제방식 컬럼 없으면 unknown
for key in ["from_bank", "to_bank", "payment_currency", "receiving_currency", "payment_format"]:
    if COL[key] is None:
        # 원본 clean_base에 없는 컬럼은 unknown 상수로 만들어 downstream 오류를 막되,
        # 실제 ML 인코딩 대상에서는 제외할 수 있도록 별도 기록합니다.
        df = df.with_columns(pl.lit("unknown").alias(key))
        COL[key] = key
        CREATED_UNKNOWN_CATEGORICAL_COLS.append(key)

# split 라벨을 원본 row에 붙여 이후 산출물에서 공통으로 사용합니다.
df = df.with_columns([
    pl.col(COL["tx_id"]).cast(pl.Utf8).alias("tx_id"),
    pl.col(COL["timestamp"]).cast(pl.Datetime).alias("timestamp"),
    pl.col(COL["label"]).cast(pl.Int64).alias("label"),
    pl.col(COL["amount"]).cast(pl.Float64).alias("amount"),

    pl.col(COL["sender_account"]).cast(pl.Utf8).alias("sender_account"),
    pl.col(COL["receiver_account"]).cast(pl.Utf8).alias("receiver_account"),

    pl.col(COL["from_bank"]).cast(pl.Utf8).fill_null("unknown").alias("from_bank"),
    pl.col(COL["to_bank"]).cast(pl.Utf8).fill_null("unknown").alias("to_bank"),
    pl.col(COL["payment_currency"]).cast(pl.Utf8).fill_null("unknown").alias("payment_currency"),
    pl.col(COL["receiving_currency"]).cast(pl.Utf8).fill_null("unknown").alias("receiving_currency"),
    pl.col(COL["payment_format"]).cast(pl.Utf8).fill_null("unknown").alias("payment_format"),
])

# 02번에서도 label을 한 번 더 확인해 잘못된 입력을 막습니다.
bad_label_df = df.filter(
    pl.col("label").is_null() |
    (~pl.col("label").is_in([0, 1]))
)

invalid_label_count = bad_label_df.height
null_timestamp_count = df.filter(pl.col("timestamp").is_null()).height
null_amount_count = df.filter(pl.col("amount").is_null()).height

print("invalid_label_count:", invalid_label_count)
print("null_timestamp_count:", null_timestamp_count)
print("null_amount_count:", null_amount_count)

if invalid_label_count > 0:
    bad_label_path = CONFIG["LOCAL_OUTPUT_DIR"] / "step02_bad_label_examples.csv"
    bad_label_df.head(50).write_csv(bad_label_path)
    raise ValueError(
        "label이 null 또는 0/1 외 값입니다. rows={}, examples={}".format(
            invalid_label_count,
            bad_label_path,
        )
    )

if null_timestamp_count > 0:
    raise ValueError("timestamp null row가 있습니다.")
if null_amount_count > 0:
    raise ValueError("amount null row가 있습니다.")

df = df.sort(["timestamp", "tx_id"])

def validate_tx_id_unique_in_step02(df):
    n_rows = df.height
    n_unique = df.select(pl.col("tx_id").n_unique()).item()

    if n_rows != n_unique:
        raise AssertionError(
            "tx_id 중복 발생: rows={}, unique_tx_id={}".format(n_rows, n_unique)
        )

    print("tx_id unique check: PASS | rows={} unique_tx_id={}".format(n_rows, n_unique))


validate_tx_id_unique_in_step02(df)

record_memory("canonical_schema_ready", df)

df.select([
    "tx_id", "timestamp", "label", "amount",
    "sender_account", "receiver_account",
    "from_bank", "to_bank", "payment_currency", "receiving_currency", "payment_format"
]).head(5)


## 5. split 확정

아래 우선순위로 split을 확정합니다.

1. `clean_base.parquet`에 `split` 컬럼이 있으면 그대로 사용
2. `split_assignment.csv`가 있으면 tx_id 기준으로 복원하여 사용
3. 기존 단일 산출물 `ml_exp00.parquet`에 `split` 컬럼이 있으면 tx_id 기준으로 복원하여 사용
4. 구버전 train/val/test 분리 parquet만 있으면 tx_id 기준으로 복원하여 사용
5. 위 항목이 모두 없으면 시간순 60/20/20 split을 한 번만 생성하고 `split_assignment.csv`로 저장

**part별·실험별로 split을 다시 생성하는 로직은 금지합니다.**  
ml_exp00, ml_exp00_no_time, ml_exp01은 모두 동일한 split을 사용합니다.


In [ ]:
# ============================================================
# split 확정 (기존 구조 유지 + 재사용 우선)
#
# [1순위] clean_base에 split 컬럼이 있으면 그대로 사용
# [2순위] split_assignment.csv가 있으면 tx_id 기준 복원
# [3순위] 단일 산출물 ml_exp00.parquet이 있으면 tx_id 기준 복원
# [4순위] 구버전 ml_exp00_train/val/test.parquet이 있으면 tx_id 기준 복원
# [5순위] 모두 없으면 시간순 60/20/20 split을 한 번만 생성
#         -> split_assignment.csv 저장 (이후 재실행 시 2순위로 재사용)
#
# 주의:
#   - part별 / 실험별 split 재생성 금지
#   - ml_exp00 / ml_exp00_no_time / ml_exp01 모두 동일 split 사용
# ============================================================

import polars as pl
import numpy as np

# ── 시간순 split 생성 함수 (5순위에서만 호출) ──────────────────────
def _make_time_split_once(df):
    """시간순 60/20/20 split. 같은 timestamp는 같은 split에 배치."""
    if 'timestamp' not in df.columns:
        raise ValueError('[SPLIT] timestamp 컬럼이 없어 시간순 split 생성 불가')
    n = df.height
    train_target = n * CONFIG['TRAIN_RATIO']
    val_target   = n * (CONFIG['TRAIN_RATIO'] + CONFIG['VAL_RATIO'])
    ts_counts = (
        df.group_by('timestamp').agg(pl.len().alias('n_rows'))
          .sort('timestamp')
          .with_columns(pl.col('n_rows').cum_sum().alias('cum_rows'))
    )
    cum = ts_counts['cum_rows'].to_numpy()
    timestamps = ts_counts['timestamp'].to_list()
    if len(timestamps) < 3:
        raise ValueError('[SPLIT] timestamp 종류가 3개 미만 — split 생성 불가')

    def _boundary(target):
        idx = int(np.searchsorted(cum, target, side='left'))
        idx = max(1, min(idx, len(timestamps) - 1))
        if idx > 0 and abs(cum[idx-1]-target) <= abs(cum[idx]-target):
            return idx
        return idx + 1 if idx < len(timestamps)-1 else idx

    tr_end = max(1, min(_boundary(train_target), len(timestamps)-2))
    va_end = max(tr_end+1, min(_boundary(val_target), len(timestamps)-1))
    print(f'[SPLIT NEW] train_last_ts={timestamps[tr_end-1]}')
    print(f'[SPLIT NEW] val_last_ts  ={timestamps[va_end-1]}')
    return df.with_columns(
        pl.when(pl.col('timestamp') <= timestamps[tr_end-1]).then(pl.lit('train'))
          .when(pl.col('timestamp') <= timestamps[va_end-1]).then(pl.lit('val'))
          .otherwise(pl.lit('test'))
          .alias('split')
    )

# ── split 유효성 체크 함수 ──────────────────────────────────────────
def _is_valid_split(df):
    if 'split' not in df.columns:
        return False
    if df.filter(pl.col('split').is_null()).height > 0:
        return False
    vals = set(df['split'].unique().to_list())
    return vals == {'train', 'val', 'test'}

def _restore_split_by_tx_id(df, restore_df, source_name):
    """restore_df(tx_id, split)를 df에 merge하고 검증한다."""
    need_cols = {'tx_id', 'split'}
    if not need_cols <= set(restore_df.columns):
        raise ValueError(f'[SPLIT ERROR] {source_name}에 tx_id 또는 split 컬럼이 없습니다.')
    restore_df = restore_df.select([
        pl.col('tx_id').cast(pl.Utf8),
        pl.col('split').cast(pl.Utf8),
    ])
    bad_vals = set(restore_df['split'].unique().to_list()) - {'train', 'val', 'test'}
    if bad_vals:
        raise ValueError(f'[SPLIT ERROR] {source_name} split 값 이상: {sorted(bad_vals)}')
    if 'split' in df.columns:
        df = df.drop('split')
    df = df.with_columns(pl.col('tx_id').cast(pl.Utf8))
    df = df.join(restore_df, on='tx_id', how='left')
    n_null = df.filter(pl.col('split').is_null()).height
    if n_null > 0:
        raise ValueError(
            f'[SPLIT ERROR] {source_name} join 후 split null {n_null:,}건 — tx_id 불일치 가능성\n'
            '해결 방법:\n'
            '1. 현재 clean_base가 기존 산출물을 만들 때 사용한 clean_base와 같은지 확인하세요.\n'
            '2. 기존 ml_exp00*.parquet가 오래된 산출물이라면 백업/삭제 후 재실행하세요.\n'
            '3. 새 split을 생성하려면 기존 ml_exp00*.parquet와 split_assignment.csv를 모두 제거한 뒤 재실행하세요.'
        )
    return df

_SPLIT_ASSIGN_PATH = CONFIG['LOCAL_OUTPUT_DIR'] / 'split_assignment.csv'
_SPLIT_SOURCE = None

# ── [1순위] clean_base에 split 컬럼이 있는 경우 ────────────────────
if _is_valid_split(df):
    _SPLIT_SOURCE = 'clean_base_column'
    print('[SPLIT 1] clean_base에 split 컬럼이 있습니다. 그대로 사용합니다.')

# ── [2순위] split_assignment.csv가 있는 경우 ───────────────────────
elif _SPLIT_ASSIGN_PATH.exists():
    print('[SPLIT 2] split_assignment.csv 발견 → tx_id 기준으로 split 복원합니다.')
    _assign_df = pl.read_csv(_SPLIT_ASSIGN_PATH)
    df = _restore_split_by_tx_id(df, _assign_df, 'split_assignment.csv')
    _SPLIT_SOURCE = 'split_assignment_csv'

# ── [3순위] 단일 산출물 ml_exp00.parquet이 있는 경우 ───────────────
elif (CONFIG['DRIVE_OUTPUT_DIR'] / 'ml_exp00.parquet').exists():
    print('[SPLIT 3] 단일 ml_exp00.parquet 발견 → tx_id 기준으로 split 복원합니다.')
    _pq = CONFIG['DRIVE_OUTPUT_DIR'] / 'ml_exp00.parquet'
    _restore_df = pl.read_parquet(_pq, columns=['tx_id', 'split'])
    df = _restore_split_by_tx_id(df, _restore_df, 'ml_exp00.parquet')
    _SPLIT_SOURCE = 'existing_ml_exp00_full_parquet'

# ── [4순위] 구버전 train/val/test 분리 parquet이 있는 경우 ─────────
elif all((CONFIG['DRIVE_OUTPUT_DIR'] / f'ml_exp00_{s}.parquet').exists()
         for s in ['train', 'val', 'test']):
    print('[SPLIT 4] 구버전 ml_exp00 train/val/test parquet 발견 → tx_id 기준으로 split 복원합니다.')
    _split_frames = []
    for _sn in ['train', 'val', 'test']:
        _pq = CONFIG['DRIVE_OUTPUT_DIR'] / f'ml_exp00_{_sn}.parquet'
        _sub = pl.read_parquet(_pq, columns=['tx_id'])
        _sub = _sub.with_columns([
            pl.col('tx_id').cast(pl.Utf8),
            pl.lit(_sn).alias('split'),
        ])
        _split_frames.append(_sub)
    _restore_df = pl.concat(_split_frames)
    df = _restore_split_by_tx_id(df, _restore_df, 'ml_exp00_train/val/test.parquet')
    _SPLIT_SOURCE = 'existing_ml_exp00_split_parquets'

# ── [5순위] 기존 split 산출물이 없으면 시간순 split 한 번 생성 ─────
else:
    print('[SPLIT 5] 기존 split 없음 → 03번에서 시간순 split을 한 번 생성합니다.')
    df = _make_time_split_once(df)
    _SPLIT_SOURCE = 'new_time_split_once'

# ── 최종 검증 + 출력 ─────────────────────────────────────────────────
if not _is_valid_split(df):
    raise ValueError('[SPLIT ERROR] split 확정 후에도 train/val/test 유효성 검증 실패')

split_summary = (
    df.group_by('split')
      .agg([
          pl.len().alias('n_rows'),
          pl.col('timestamp').min().alias('timestamp_min'),
          pl.col('timestamp').max().alias('timestamp_max'),
          pl.col('label').sum().alias('n_positive') if 'label' in df.columns else pl.lit(None).alias('n_positive'),
      ])
      .with_columns((pl.col('n_positive') / pl.col('n_rows')).alias('positive_ratio'))
      .sort(pl.col('split').replace({'train': 0, 'val': 1, 'test': 2}))
)

print(f'[SPLIT OK] source={_SPLIT_SOURCE}')
display(split_summary.to_pandas())


In [ ]:
# ============================================================
# split 심화 검증
# - tx_id 단위 split 유일성 확인
# - 시간 단조성 확인 (train < val < test)
# - ml_exp00 / ml_exp00_no_time / ml_exp01 동일 split 보장 선언
# ============================================================

# tx_id 단위 유일성
_tx_multi = (
    df.group_by('tx_id').agg(pl.col('split').n_unique().alias('n_split'))
      .filter(pl.col('n_split') > 1).height
)
if _tx_multi > 0:
    raise ValueError(f'[SPLIT ERROR] tx_id가 여러 split에 걸쳐 있음: {_tx_multi}건')

# 시간 단조성: train max ≤ val min, val max ≤ test min
_ts = {
    row['split']: row
    for row in df.group_by('split').agg(
        pl.col('timestamp').min().alias('ts_min'),
        pl.col('timestamp').max().alias('ts_max'),
    ).to_dicts()
}
if all(s in _ts for s in ['train', 'val', 'test']):
    assert _ts['train']['ts_max'] <= _ts['val']['ts_min'], (
        f'[SPLIT ERROR] train max > val min: '
        f'{_ts["train"]["ts_max"]} > {_ts["val"]["ts_min"]}'
    )
    assert _ts['val']['ts_max'] <= _ts['test']['ts_min'], (
        f'[SPLIT ERROR] val max > test min: '
        f'{_ts["val"]["ts_max"]} > {_ts["test"]["ts_min"]}'
    )
    print('[SPLIT OK] 시간 단조성 확인: train ≤ val ≤ test')

# split_assignment.csv 저장 (1순위·3순위·4순위 모두 동일하게 보존)
# 이미 저장된 경우에도 현재 df의 split과 일치하는지 보장하기 위해 갱신
if _SPLIT_SOURCE != 'split_assignment_csv':  # 2순위는 이미 CSV에서 읽었으므로 스킵
    df.select(['tx_id', 'split']).write_csv(_SPLIT_ASSIGN_PATH)
    print(f'[SPLIT] split_assignment.csv 갱신: {_SPLIT_ASSIGN_PATH}')

print(f'[SPLIT DEEP OK] source={_SPLIT_SOURCE}')
print('ml_exp00 / ml_exp00_no_time / ml_exp01 모두 이 split을 공유합니다.')


## 6. split summary & baseline feature 구성

split별 row 수, 라벨 분포, timestamp 범위를 기록합니다.  
이어서 현재 거래에서 바로 추출할 수 있는 baseline feature를 구성합니다.

In [ ]:
# ============================================================
# split_summary.csv
# ============================================================

# split별 row 수, 라벨 분포, timestamp 범위를 요약합니다.
def make_split_summary(df):
    summary = (
        df.group_by("split")
        .agg([
            pl.len().alias("row_count"),
            (pl.col("label") == 0).sum().alias("label_0_count"),
            (pl.col("label") == 1).sum().alias("label_1_count"),
            pl.col("label").mean().alias("label_1_ratio"),
            pl.col("timestamp").min().alias("timestamp_min"),
            pl.col("timestamp").max().alias("timestamp_max"),
        ])
    )

    order = pl.DataFrame({
        "split": ["train", "val", "test"],
        "split_order": [0, 1, 2],
    })

    return (
        summary.join(order, on="split", how="left")
        .sort("split_order")
        .drop("split_order")
    )


# split 결과는 별도 CSV로 저장합니다.
split_summary = make_split_summary(df)
split_summary_path = CONFIG["LOCAL_OUTPUT_DIR"] / "split_summary.csv"
split_summary.write_csv(split_summary_path)

print("saved:", split_summary_path)
split_summary


## 7. time feature 포함 버전 생성
시간대, 요일 등 시간 기반 피처가 성능에 도움이 되는지 비교하기 위한 버전입니다. 최종 사용 여부는 baseline 결과를 보고 판단합니다.

In [ ]:
# ============================================================
# ML-00: 현재 거래 feature
# [컬럼명 원칙] 기존 전처리파트 output 컬럼명을 그대로 유지합니다.
# ML파트 공식 표시명은 feature_catalog.csv에 alias로만 기록됩니다.
# 추가 피처(amount_received, ratio)는 기존 명명 규칙을 따릅니다.
# ============================================================

# [QA Fix] polars dt.weekday() 반환값 확인
# polars: 1=월, 2=화, ..., 6=토, 7=일 (ISO 8601)
# 코드에서 0=월~6=일로 맞추기 위해 -1을 적용합니다.
# is_weekend: 토(5) 또는 일(6) -> weekday-1 >= 5

def add_ml00_features(df):
    exprs = [
        # ── 기존 컬럼 (전처리파트 output 원본 유지) ──────────────────────────
        pl.col('amount').alias('amount__current__value'),
        safe_log1p_expr('amount').alias('amount__current__log1p'),
        pl.col('timestamp').dt.hour().cast(pl.Int16).alias('time__row__hour'),
        # dt.weekday()-1: 0=월, 1=화, 2=수, 3=목, 4=금, 5=토, 6=일
        (pl.col('timestamp').dt.weekday() - 1).cast(pl.Int16).alias('time__row__dayofweek'),
        # is_weekend: 토(5) 또는 일(6)
        ((pl.col('timestamp').dt.weekday() - 1) >= 5).cast(pl.Int8).alias('time__row__is_weekend'),
    ]

    # ── 추가 피처: amount_received (있으면 생성, 없으면 건너뜀) ──────────
    if 'amount_received' in df.columns:
        exprs += [
            pl.col('amount_received').cast(pl.Float64).alias('amount_received__current__value'),
            safe_log1p_expr('amount_received').alias('amount_received__current__log1p'),
        ]
        exprs.append(
            pl.when(
                pl.col('amount_received').is_null() |
                (pl.col('amount_received').cast(pl.Float64) == 0.0)
            ).then(None)
             .otherwise(pl.col('amount').cast(pl.Float64) / pl.col('amount_received').cast(pl.Float64))
             .alias('amount__paid_recv_ratio')
        )
    else:
        print('[INFO] amount_received 컬럼 없음 -> amount_received 계열 피처 생성 건너뜀.')

    return df.with_columns(exprs)


df_ml00_base = add_ml00_features(df)
record_memory('ml00_base_features_created', df_ml00_base)

# [QA Fix] weekday 값 범위 검증
_wd_vals = df_ml00_base['time__row__dayofweek'].drop_nulls().unique().sort().to_list()
assert all(0 <= v <= 6 for v in _wd_vals), f'time__row__dayofweek 범위 이상: {_wd_vals}'
_we_vals = set(df_ml00_base['time__row__is_weekend'].drop_nulls().unique().to_list())
assert _we_vals <= {0, 1}, f'time__row__is_weekend 값 이상: {_we_vals}'
print(f'[OK] time__row__dayofweek range: {min(_wd_vals)}~{max(_wd_vals)} (0=월,6=일)')
print(f'[OK] time__row__is_weekend values: {sorted(_we_vals)}')

_preview = [
    'tx_id', 'timestamp', 'split', 'label',
    'amount__current__value', 'amount__current__log1p',
    'time__row__hour', 'time__row__dayofweek', 'time__row__is_weekend',
    'from_bank', 'to_bank', 'payment_currency', 'receiving_currency', 'payment_format'
]
df_ml00_base.select([c for c in _preview if c in df_ml00_base.columns]).head(5)


## 8. 범주형 피처 인코딩

ML파트 전달용 기본 산출물에 맞춰 categorical feature는 train split에서만 fit한 label/code 인코딩으로 추가합니다.

- 기존 `ml_exp00`, `ml_exp00_no_time`, `ml_exp01` 구조는 유지합니다.
- one-hot/hybrid 비교용 산출물은 만들지 않습니다.
- validation/test에서 train에 없던 category는 `-1`로 처리합니다.
- 원문 계좌 식별자, label/split/timestamp, pattern/laundering 관련 정보는 모델 입력 feature에서 제외합니다.

In [ ]:
# ============================================================
# 범주형 인코딩
# - fit은 train 데이터만 사용한다.
# - val/test 신규 category는 label/code -1로 처리한다.
# - ML파트 전달용 최종 산출물은 기존 구조 + label/code encoded columns만 생성한다.
# ============================================================

CATEGORICAL_CANDIDATE_COLS = [
    "from_bank",
    "to_bank",
    "payment_currency",
    "receiving_currency",
    "payment_format",
]

RAW_IDENTIFIER_COLS_EXCLUDED_FROM_ML = [
    "tx_id",
    "sender_account",
    "receiver_account",
    "account_id",
    "sender_account_id",
    "receiver_account_id",
    "Account",
    "Account.1",
]

LEAKAGE_LIKE_COLS_EXCLUDED_FROM_ML = [
    "label",
    "Is_Laundering",
    "Is Laundering",
    "split",
    "timestamp",
    "Patterns.txt",
    "laundering_attempt_id",
    "pattern_type",
]

created_unknown_cols = set(globals().get("CREATED_UNKNOWN_CATEGORICAL_COLS", []))

# 실제 clean_base에서 확인 가능한 categorical 후보만 사용합니다.
# 원본에 없어서 unknown 상수로 만든 컬럼은 모델 신호가 아니므로 제외합니다.
CATEGORICAL_RAW_COLS = [
    c for c in CATEGORICAL_CANDIDATE_COLS
    if c in df_ml00_base.columns and c not in created_unknown_cols
]

CATEGORICAL_EXCLUDED_RECORDS = []
for c in CATEGORICAL_CANDIDATE_COLS:
    if c not in df_ml00_base.columns:
        CATEGORICAL_EXCLUDED_RECORDS.append({
            "column_name": c,
            "excluded_reason": "column_not_found_in_clean_base",
            "leakage_rule": "not_used",
        })
    elif c in created_unknown_cols:
        CATEGORICAL_EXCLUDED_RECORDS.append({
            "column_name": c,
            "excluded_reason": "created_as_constant_unknown_because_source_column_missing",
            "leakage_rule": "not_used_as_model_signal",
        })

for c in RAW_IDENTIFIER_COLS_EXCLUDED_FROM_ML:
    if c in df_ml00_base.columns:
        CATEGORICAL_EXCLUDED_RECORDS.append({
            "column_name": c,
            "excluded_reason": "raw_account_or_transaction_identifier",
            "leakage_rule": "model_input_excluded_P0",
        })

for c in LEAKAGE_LIKE_COLS_EXCLUDED_FROM_ML:
    if c in df_ml00_base.columns:
        CATEGORICAL_EXCLUDED_RECORDS.append({
            "column_name": c,
            "excluded_reason": "label_split_timestamp_or_label_like_information",
            "leakage_rule": "model_input_excluded_P0",
        })

categorical_excluded_df = pd.DataFrame(
    CATEGORICAL_EXCLUDED_RECORDS,
    columns=["column_name", "excluded_reason", "leakage_rule"],
).drop_duplicates()

categorical_excluded_path = CONFIG["LOCAL_OUTPUT_DIR"] / "categorical_excluded_columns.csv"
categorical_excluded_df.to_csv(categorical_excluded_path, index=False, encoding="utf-8-sig")
print("saved:", categorical_excluded_path)
print("CATEGORICAL_RAW_COLS:", CATEGORICAL_RAW_COLS)

CAT_FEATURE_MAP = {
    raw_col: "cat__{}__code".format(raw_col)
    for raw_col in CATEGORICAL_RAW_COLS
}


def _normalized_string_expr(col):
    return pl.col(col).cast(pl.Utf8).fill_null("unknown")


def _unique_values(df, col, split_name=None):
    work = df
    if split_name is not None:
        work = work.filter(pl.col("split") == split_name)
    if work.height == 0:
        return set()
    return set(
        work.select(_normalized_string_expr(col).alias(col))
        .unique()
        .to_series()
        .to_list()
    )


def _n_rows_not_in_train(df, col, split_name, train_values):
    part = df.filter(pl.col("split") == split_name)
    if part.height == 0:
        return 0
    return part.filter(~_normalized_string_expr(col).is_in(list(train_values))).height


def fit_label_maps_train_only(df, cols):
    train_df = df.filter(pl.col("split") == "train")
    if train_df.height == 0:
        raise ValueError("train split이 비어 있어 categorical encoder를 fit할 수 없습니다.")

    encoders = {}

    for col in cols:
        vc = (
            train_df
            .select(_normalized_string_expr(col).alias(col))
            .group_by(col)
            .agg(pl.len().alias("train_count"))
            .sort(col)
        )

        values = [str(v) for v in vc[col].to_list()]
        counts = [int(v) for v in vc["train_count"].to_list()]
        label_map = {category: i for i, category in enumerate(values)}
        count_map = dict(zip(values, counts))
        ratio_map = {
            category: count_map[category] / train_df.height
            for category in values
        }

        encoders[col] = {
            "label_map": label_map,
            "train_count": count_map,
            "train_ratio": ratio_map,
            "train_values": set(values),
        }
        print("[fit:label_code]", col, "n_unique_train=", len(values))

    return encoders


def apply_label_maps(df, encoders):
    out = df

    for raw_col, encoder in encoders.items():
        feature_col = CAT_FEATURE_MAP[raw_col]
        mapping = encoder["label_map"]

        map_df = pl.DataFrame({
            raw_col: list(mapping.keys()),
            feature_col: list(mapping.values()),
        }).with_columns([
            pl.col(raw_col).cast(pl.Utf8),
            pl.col(feature_col).cast(pl.Int32),
        ])

        out = (
            out.with_columns(_normalized_string_expr(raw_col).alias(raw_col))
            .join(map_df, on=raw_col, how="left")
            .with_columns(pl.col(feature_col).fill_null(-1).cast(pl.Int32))
        )

    return out


# category encoder는 train split만 기준으로 fit합니다.
CATEGORY_ENCODERS = fit_label_maps_train_only(df_ml00_base, CATEGORICAL_RAW_COLS)
df_ml00_base = apply_label_maps(df_ml00_base, CATEGORY_ENCODERS)

CAT_LABEL_FEATURE_COLS = [CAT_FEATURE_MAP[c] for c in CATEGORICAL_RAW_COLS]

# train-only mapping report
cat_map_records = []
for raw_col, encoder in CATEGORY_ENCODERS.items():
    label_map = encoder["label_map"]
    for category, code in label_map.items():
        train_count = encoder["train_count"].get(category, 0)
        train_ratio = encoder["train_ratio"].get(category, 0.0)

        cat_map_records.append({
            "raw_column": raw_col,
            "encoding_type": "label_code",
            "feature_column": CAT_FEATURE_MAP[raw_col],
            "category": category,
            "code": code,
            "train_count": train_count,
            "train_ratio": train_ratio,
            "fit_scope": "train_only",
            "unknown_policy": "val/test unseen category -> -1",
        })

category_mapping_train_only_df = pd.DataFrame(
    cat_map_records,
    columns=[
        "raw_column",
        "encoding_type",
        "feature_column",
        "category",
        "code",
        "train_count",
        "train_ratio",
        "fit_scope",
        "unknown_policy",
    ],
)
category_map_path = CONFIG["LOCAL_OUTPUT_DIR"] / "category_mapping_train_only.csv"
category_mapping_train_only_df.to_csv(category_map_path, index=False, encoding="utf-8-sig")
print("saved:", category_map_path)

# 인코딩 전략/컬럼 증가 규모/unknown category 현황 요약
summary_records = []
for raw_col in CATEGORICAL_RAW_COLS:
    train_values = CATEGORY_ENCODERS[raw_col]["train_values"]
    val_values = _unique_values(df_ml00_base, raw_col, "val")
    test_values = _unique_values(df_ml00_base, raw_col, "test")
    all_values = _unique_values(df_ml00_base, raw_col, None)

    summary_records.append({
        "raw_column": raw_col,
        "selected_encoding": "train_only_label_code",
        "encoded_feature_column": CAT_FEATURE_MAP[raw_col],
        "n_added_encoded_columns": 1,
        "n_unique_train": len(train_values),
        "n_unique_val": len(val_values),
        "n_unique_test": len(test_values),
        "n_unique_all": len(all_values),
        "n_new_val_categories_not_in_train": len(val_values - train_values),
        "n_new_test_categories_not_in_train": len(test_values - train_values),
        "n_new_val_rows_not_in_train": _n_rows_not_in_train(df_ml00_base, raw_col, "val", train_values),
        "n_new_test_rows_not_in_train": _n_rows_not_in_train(df_ml00_base, raw_col, "test", train_values),
        "unknown_policy": "-1 for val/test unseen category",
        "target_encoding_used": False,
        "selection_reason": "ML파트 전달용 기존 산출물 구조 유지: categorical column만 train-only label/code로 추가",
    })

categorical_encoding_summary = pd.DataFrame(summary_records)
cat_summary_path = CONFIG["LOCAL_OUTPUT_DIR"] / "categorical_encoding_summary.csv"
categorical_encoding_summary.to_csv(cat_summary_path, index=False, encoding="utf-8-sig")
print("saved:", cat_summary_path)

print("label/code categorical features:", len(CAT_LABEL_FEATURE_COLS))

record_memory("categorical_encoded_train_only", df_ml00_base)

show_cols = ["split"]
for raw_col in CATEGORICAL_RAW_COLS:
    show_cols += [raw_col, CAT_FEATURE_MAP[raw_col]]

display(categorical_encoding_summary)
df_ml00_base.select(show_cols).head(5) if show_cols != ["split"] else df_ml00_base.select(show_cols).head(5)


# ============================================================
# [오늘 회의 후보] Pair Encoding (optional candidate)
# 오늘 회의에서 후보로 언급 -> 기존 구조를 깨지 않는 범위에서 생성
# 컬럼명은 기존 cat__*__code 규칙 동일하게 적용
# 최종 사용 여부는 ml_feature_columns.csv used_in_ml=True/False로 결정
# ============================================================

PAIR_ENCODING_DEFS = {
    'cat__currency_pair__code': ('payment_currency', 'receiving_currency'),
    'cat__bank_pair__code':     ('from_bank', 'to_bank'),
}

def _pair_key(pair_col): return f'__praw__{pair_col}'

def add_raw_pair_cols(df):
    exprs = []
    for pair_col, (col_a, col_b) in PAIR_ENCODING_DEFS.items():
        if col_a in df.columns and col_b in df.columns:
            exprs.append(
                (pl.col(col_a).cast(pl.Utf8).fill_null('unknown') + pl.lit('||') +
                 pl.col(col_b).cast(pl.Utf8).fill_null('unknown')).alias(_pair_key(pair_col))
            )
        else:
            print(f'[SKIP] pair {pair_col}: 원본 컬럼 없음 ({col_a}, {col_b})')
    return df.with_columns(exprs) if exprs else df

def fit_pair_maps(df, pair_defs):
    train_df = df.filter(pl.col('split') == 'train')
    encoders = {}
    for pair_col, (col_a, col_b) in pair_defs.items():
        raw_col = _pair_key(pair_col)
        if raw_col not in df.columns: continue
        vals = (train_df.select(pl.col(raw_col)).group_by(raw_col)
                        .agg(pl.len()).sort(raw_col)[raw_col].to_list())
        encoders[pair_col] = {'map': {v:i for i,v in enumerate(vals)}, 'raw_col': raw_col}
        print(f'[fit:pair_code] {pair_col} n_train={len(vals)}')
    return encoders

def apply_pair_maps(df, encoders):
    out = df
    for pair_col, enc in encoders.items():
        raw_col = enc['raw_col']
        m = enc['map']
        map_df = pl.DataFrame({raw_col: list(m.keys()), pair_col: list(m.values())})
        map_df = map_df.with_columns([pl.col(raw_col).cast(pl.Utf8), pl.col(pair_col).cast(pl.Int32)])
        out = (out.join(map_df, on=raw_col, how='left')
                  .with_columns(pl.col(pair_col).fill_null(-1).cast(pl.Int32)))
    return out

df_ml00_base = add_raw_pair_cols(df_ml00_base)
PAIR_ENCODERS = fit_pair_maps(df_ml00_base, PAIR_ENCODING_DEFS)
df_ml00_base = apply_pair_maps(df_ml00_base, PAIR_ENCODERS)

PAIR_FEATURE_COLS = list(PAIR_ENCODERS.keys())
print('pair feature cols (candidate):', PAIR_FEATURE_COLS)
print('-> used_in_ml 기본값: False (ml_feature_columns.csv에서 True로 바꿔야 활성화)')

_ps = []
for pc, enc in PAIR_ENCODERS.items():
    raw_col = enc['raw_col']
    tr_keys = set(enc['map'].keys())
    va_keys = set(df_ml00_base.filter(pl.col('split')=='val' ).select(raw_col).to_series().to_list()) if raw_col in df_ml00_base.columns else set()
    te_keys = set(df_ml00_base.filter(pl.col('split')=='test').select(raw_col).to_series().to_list()) if raw_col in df_ml00_base.columns else set()
    _ps.append({'pair_feature_col':pc,'n_train':len(tr_keys),'n_new_val':len(va_keys-tr_keys),'n_new_test':len(te_keys-tr_keys),'used_in_ml_default':'False (candidate)'})
pd.DataFrame(_ps).to_csv(CONFIG['LOCAL_OUTPUT_DIR']/'pair_encoding_summary.csv', index=False, encoding='utf-8-sig')
print('saved: pair_encoding_summary.csv')
record_memory('pair_encoded', df_ml00_base)


## 9. Stage 0 safe history feature 생성
누수 방지를 가장 우선으로 두고, `timestamp < current_timestamp` 조건의 과거 거래만 사용합니다. 현재 row와 동일 timestamp 거래는 과거로 보지 않습니다.

In [ ]:
# 누수 방지 기준이 가장 중요한 history feature
# ============================================================
# Stage 0 history feature
# 조건: current_timestamp - window <= past_timestamp < current_timestamp
# 동일 timestamp 거래는 과거로 보지 않는다.
# ============================================================

def compute_entity_history_features(
    df,
    entity_col,
    amount_col,
    timestamp_col,
    direction_label,
    entity_scope_label,
    windows,
):
    work = df.select(["tx_id", entity_col, amount_col, timestamp_col]).sort(
        [entity_col, timestamp_col, "tx_id"]
    )

    tx_ids = work["tx_id"].to_list()
    entities = work[entity_col].to_list()
    amounts = work[amount_col].to_numpy()
    ts_ns = series_datetime_to_ns(work[timestamp_col])

    n = work.height
    result = {"tx_id": tx_ids}

    for w_name in windows.keys():
        result["timehist__{}__{}__tx_count__count__{}".format(
            entity_scope_label, direction_label, w_name
        )] = np.zeros(n, dtype=np.int64)

        result["timehist__{}__{}__amount__sum__{}".format(
            entity_scope_label, direction_label, w_name
        )] = np.zeros(n, dtype=np.float64)

        result["timehist__{}__{}__amount__mean__{}".format(
            entity_scope_label, direction_label, w_name
        )] = np.zeros(n, dtype=np.float64)

    # timestamp라는 토큰은 금지 컬럼 검사에서 걸리므로 피처명에 넣지 않는다.
    recency_col = "recency__{}__{}__seconds_since_last".format(
        entity_scope_label, direction_label
    )
    first_col = "flag__{}__{}__is_first_tx".format(
        entity_scope_label, direction_label
    )

    result[recency_col] = np.full(n, np.nan, dtype=np.float64)
    result[first_col] = np.zeros(n, dtype=np.int8)

    entity_to_indices = {}
    for i, ent in enumerate(entities):
        entity_to_indices.setdefault(ent, []).append(i)

    for ent, idx_list in entity_to_indices.items():
        idx_arr = np.array(idx_list, dtype=np.int64)
        ent_ts = ts_ns[idx_arr]
        ent_amount = amounts[idx_arr].astype(float)
        prefix_amount = np.concatenate([[0.0], np.cumsum(ent_amount)])

        for local_pos, global_idx in enumerate(idx_arr):
            current_ts = ent_ts[local_pos]

            # side='left'라서 같은 timestamp 거래는 제외된다.
            end = np.searchsorted(ent_ts, current_ts, side="left")

            if end == 0:
                result[first_col][global_idx] = 1
                result[recency_col][global_idx] = np.nan
            else:
                last_ts = ent_ts[end - 1]
                result[recency_col][global_idx] = (current_ts - last_ts) / 1_000_000_000

            for w_name, w_str in windows.items():
                window_ns = parse_window_to_ns(w_str)
                start_ts = current_ts - window_ns
                start = np.searchsorted(ent_ts, start_ts, side="left")

                count = end - start
                amount_sum = prefix_amount[end] - prefix_amount[start]
                amount_mean = amount_sum / count if count > 0 else 0.0

                count_col = "timehist__{}__{}__tx_count__count__{}".format(
                    entity_scope_label, direction_label, w_name
                )
                sum_col = "timehist__{}__{}__amount__sum__{}".format(
                    entity_scope_label, direction_label, w_name
                )
                mean_col = "timehist__{}__{}__amount__mean__{}".format(
                    entity_scope_label, direction_label, w_name
                )

                result[count_col][global_idx] = count
                result[sum_col][global_idx] = amount_sum
                result[mean_col][global_idx] = amount_mean

    return pl.DataFrame(result)


windows = dict(CONFIG["WINDOWS"])
if CONFIG["MAKE_OPTIONAL_30D"]:
    windows["optional_30d"] = "30d"

print("windows:", windows)

sender_hist = compute_entity_history_features(
    df=df_ml00_base,
    entity_col="sender_account",
    amount_col="amount",
    timestamp_col="timestamp",
    direction_label="out",
    entity_scope_label="sender",
    windows=windows,
)
record_memory("sender_history_created", sender_hist)

receiver_hist = compute_entity_history_features(
    df=df_ml00_base,
    entity_col="receiver_account",
    amount_col="amount",
    timestamp_col="timestamp",
    direction_label="in",
    entity_scope_label="receiver",
    windows=windows,
)
record_memory("receiver_history_created", receiver_hist)

print_shape("sender_hist", sender_hist)
print_shape("receiver_hist", receiver_hist)


## 10. ML-01 구성 (history feature 결합)

baseline feature + history feature를 `tx_id` 기준으로 결합해 ml_exp01 입력을 구성합니다.

In [ ]:
# ============================================================
# ML-01 구성
# ============================================================

# baseline feature와 history feature를 tx_id 기준으로 결합합니다.
df_ml01_base = (
    df_ml00_base
    .join(sender_hist, on="tx_id", how="left")
    .join(receiver_hist, on="tx_id", how="left")
)

fill_zero_cols = [
    c for c in df_ml01_base.columns
    if c.startswith("timehist__") or c.startswith("flag__")
]

df_ml01_base = df_ml01_base.with_columns([
    pl.col(c).fill_null(0) for c in fill_zero_cols
])

record_memory("ml01_base_created", df_ml01_base)
print_shape("df_ml01_base", df_ml01_base)

history_cols_preview = [c for c in df_ml01_base.columns if c.startswith("timehist__")][:10]
recency_cols_preview = [c for c in df_ml01_base.columns if c.startswith("recency__")]

df_ml01_base.select(
    ["tx_id", "timestamp", "split", "label"] + history_cols_preview + recency_cols_preview
).head(5)


## 11. 모델 입력 feature 목록 정의

`ML00_FEATURE_COLS` / `ML01_FEATURE_COLS` 등 각 실험의 feature 목록을 확정합니다.  
pair / amount_received 후보 컬럼은 parquet에는 포함하되 `used_in_ml=False` 기본값으로 관리합니다.

In [ ]:
# ============================================================
# 모델 입력 feature 목록
# [원칙] 기존 전처리파트 output 컬럼명 유지.
# [QA-A안] ml_feature_columns.csv 기본값은 ml_exp00 전용 feature만 True
#          history(ml_exp01) → used_in_ml=False
#          amount_received 계열 → add_candidate (used_in_ml=False)
#          pair feature → candidate (used_in_ml=False, parquet에 포함)
# ============================================================

META_COLS = ['tx_id','timestamp','split','label','sender_account','receiver_account']

# ── ml_exp00 기본 feature (used_in_ml=True 대상) ──────────────────────
AMOUNT_BASE_COLS = ['amount__current__value', 'amount__current__log1p']
TIME_FEATURE_COLS = ['time__row__hour','time__row__dayofweek','time__row__is_weekend']
CAT_FEATURE_COLS_ALL = CAT_LABEL_FEATURE_COLS  # 5개 기본

ML00_FEATURE_COLS         = AMOUNT_BASE_COLS + CAT_FEATURE_COLS_ALL + TIME_FEATURE_COLS
ML00_NO_TIME_FEATURE_COLS = AMOUNT_BASE_COLS + CAT_FEATURE_COLS_ALL

# ── amount_received 계열: add_candidate (parquet에 존재 시 활성화 가능) ─
# ML00_FEATURE_COLS에는 포함하지 않음. used_in_ml=False 기본값.
AMOUNT_RECV_CANDIDATE_COLS = [
    c for c in ['amount_received__current__value',
                'amount_received__current__log1p',
                'amount__paid_recv_ratio']
    if c in df_ml00_base.columns
]
if AMOUNT_RECV_CANDIDATE_COLS:
    print('[INFO] amount_received 계열 parquet에 포함 (candidate, used_in_ml=False):', AMOUNT_RECV_CANDIDATE_COLS)
else:
    print('[INFO] amount_received 컬럼 없음 -> 관련 후보 피처 없음')

# ── pair feature: candidate (parquet에 포함, used_in_ml=False) ───────
PAIR_COLS_IN_PARQUET = [c for c in PAIR_FEATURE_COLS if c in df_ml00_base.columns]
if len(PAIR_COLS_IN_PARQUET) < len(PAIR_FEATURE_COLS):
    print(f'[WARN] pair col 미생성: {[c for c in PAIR_FEATURE_COLS if c not in df_ml00_base.columns]}')

# ── ml_exp01: history feature (used_in_ml=False 기본값) ──────────────
ML01_HISTORY_FEATURE_COLS = [
    c for c in df_ml01_base.columns
    if c.startswith('timehist__') or c.startswith('recency__') or c.startswith('flag__')
]
ML01_FEATURE_COLS = ML00_FEATURE_COLS + ML01_HISTORY_FEATURE_COLS

FEATURE_SET_REGISTRY = {
    'ml_exp00':         ML00_FEATURE_COLS,
    'ml_exp00_no_time': ML00_NO_TIME_FEATURE_COLS,
    'ml_exp01':         ML01_FEATURE_COLS,
}

print(f'ML00 feature cols    : {len(ML00_FEATURE_COLS)}')
print(f'ML00 no-time cols    : {len(ML00_NO_TIME_FEATURE_COLS)}')
print(f'ML01 history cols    : {len(ML01_HISTORY_FEATURE_COLS)}')
print(f'ML01 total cols      : {len(ML01_FEATURE_COLS)}')
print(f'amount_recv candidate: {AMOUNT_RECV_CANDIDATE_COLS}')
print(f'pair candidate       : {PAIR_COLS_IN_PARQUET} (parquet 포함, used_in_ml=False)')

for name, cols in FEATURE_SET_REGISTRY.items():
    ref_df = df_ml01_base if name == 'ml_exp01' else df_ml00_base
    bad = [c for c in cols if c not in ref_df.columns]
    if bad: print(f'[WARN] {name} missing: {bad}')


## 12. 최종 ML 데이터셋 구성

`ml_exp00`, `ml_exp00_no_time`, `ml_exp01`을 DataFrame으로 구성합니다.  
pair / amount_received 후보 컬럼은 parquet에 포함하되 ML feature list에는 넣지 않습니다.

In [ ]:
# ============================================================
# 최종 ML 데이터셋
# [QA Fix] pair feature는 parquet에 포함 (used_in_ml=False candidate)
#          ml_feature_columns.csv에서 True로 바꾸면 바로 활성화 가능
# ============================================================

def select_existing_cols(df, cols):
    return [c for c in cols if c in df.columns]

META_KEEP_COLS = select_existing_cols(df_ml01_base, META_COLS)

# pair col은 parquet에 포함하되 ML feature set(ML00_FEATURE_COLS)에는 없음
_pair_keep = select_existing_cols(df_ml00_base, PAIR_COLS_IN_PARQUET)

ml_exp00         = df_ml00_base.select(META_KEEP_COLS + ML00_FEATURE_COLS + _pair_keep)
ml_exp00_no_time = df_ml00_base.select(META_KEEP_COLS + ML00_NO_TIME_FEATURE_COLS + _pair_keep)
ml_exp01         = df_ml01_base.select(
    select_existing_cols(df_ml01_base, META_KEEP_COLS + ML01_FEATURE_COLS + _pair_keep)
)

ml_exp00_Xy         = df_ml00_base.select(['tx_id','split','label'] + ML00_FEATURE_COLS + _pair_keep)
ml_exp00_no_time_Xy = df_ml00_base.select(['tx_id','split','label'] + ML00_NO_TIME_FEATURE_COLS + _pair_keep)
ml_exp01_Xy         = df_ml01_base.select(
    select_existing_cols(df_ml01_base, ['tx_id','split','label'] + ML01_FEATURE_COLS + _pair_keep)
)

record_memory('ml_exp00_ready', ml_exp00)
record_memory('ml_exp00_no_time_ready', ml_exp00_no_time)
record_memory('ml_exp01_ready', ml_exp01)

for name, data in [
    ('ml_exp00', ml_exp00), ('ml_exp00_no_time', ml_exp00_no_time), ('ml_exp01', ml_exp01),
    ('ml_exp00_Xy', ml_exp00_Xy), ('ml_exp00_no_time_Xy', ml_exp00_no_time_Xy), ('ml_exp01_Xy', ml_exp01_Xy),
]:
    print_shape(name, data)
    if _pair_keep:
        _pc = [c for c in _pair_keep if c in data.columns]
        print(f'  pair cols in parquet: {_pc}')


## 13. feature catalog / ml_feature_columns / column_mapping_review

각 피처의 출처, 계산 방식, leakage 규칙, alias를 기록합니다.  
`ml_feature_columns.csv`는 `ml_io.load_feature_columns()`의 입력 기준입니다.  
`column_mapping_review.csv`는 전처리파트 output ↔ ML파트 공식 표시명 기준 비교 기준입니다.

In [ ]:
# ============================================================
# feature_catalog.csv
# [컬럼명 원칙] parquet_column_name = 기존 전처리파트 컬럼명 유지
# official_feature_name = ML파트 공식 표시명을 catalog에만 기록
# ============================================================

def make_feature_catalog():
    records = []

    def add(cur, disp_name, desc, src, calc, dtype, unit, tw, miss, leak, uml, notes, owner, status='selected'):
        # disp_name: ML파트 공식 피처 명세의 표준 표시명 (parquet 컬럼명과 다를 수 있음)
        records.append({
            'feature_name': cur,           # 실제 parquet 컬럼명 (전처리파트 output)
            'official_feature_name': disp_name,  # ML파트 공식 표시명 (명세 문서 기준)
            'description': desc, 'source_data': src,
            'calculation_method': calc, 'data_type': dtype,
            'unit': unit, 'time_window': tw,
            'missing_policy': miss, 'leakage_risk': leak,
            'used_in_ml': uml, 'notes': notes,
            'owner_part': owner, 'selection_status': status,
        })

    # 금액 (기존 컬럼명 유지)
    add('amount__current__value','raw__tx__cur__amt_paid__val__wcur',
        '현재 거래 지급 금액(amount 컬럼)','amount','identity','numeric','원','현재거래','비null필수','current row only',True,'기존 전처리파트 컬럼. ML파트 공식 표시명 catalog에만 기록','PART1')
    add('amount__current__log1p','raw__tx__cur__amt__log_paid__wcur',
        '지급 금액 log1p','amount','log1p','numeric','log(원+1)','현재거래','음수->null','current row only',True,'기존 전처리파트 컬럼. ML파트 공식 표시명 catalog에만 기록','PART1')
    add('amount_received__current__value','raw__tx__cur__amt_recv__val__wcur',
        '현재 거래 수취 금액','amount_received','identity','numeric','원','현재거래','비null필수','current row only',False,
        'add_candidate: 기본 used_in_ml=False. parquet에 포함됨. ml_feature_columns.csv에서 True로 변경 시 활성화 가능','PART1','add_candidate')
    add('amount_received__current__log1p','raw__tx__cur__amt__log_recv__wcur',
        '수취 금액 log1p','amount_received','log1p','numeric','log(원+1)','현재거래','음수->null','current row only',False,
        'add_candidate: 기본 used_in_ml=False. parquet에 포함됨. ml_feature_columns.csv에서 True로 변경 시 활성화 가능','PART1','add_candidate')
    add('amount__paid_recv_ratio','raw__tx__cur__amt__paid_recv_ratio__wcur',
        '지급/수취 금액 비율','amount,amount_received','paid/received','numeric','무차원','현재거래','received=0->null','current row only',False,
        'add_candidate: 기본 used_in_ml=False. parquet에 포함됨. ml_feature_columns.csv에서 True로 변경 시 활성화 가능','PART1','add_candidate')

    # 시간 (기존 컬럼명 유지)
    add('time__row__hour','raw__tx__cur__time__hour__wcur',
        '거래 시간(0-23)','timestamp','dt.hour()','numeric/int','시','현재거래','non-null','current ts only',True,'기존 전처리파트 컬럼','PART1')
    add('time__row__dayofweek','raw__tx__cur__time__dow__wcur',
        '거래 요일(0=월~6=일)','timestamp','dt.weekday()','numeric/int','요일코드','현재거래','non-null','current ts only',True,'기존 전처리파트 컬럼','PART1')
    add('time__row__is_weekend','raw__tx__cur__time__is_weekend__wcur',
        '주말여부(1=주말)','timestamp','weekday>=6->1','numeric/int','0/1','현재거래','non-null','current ts only',True,'기존 전처리파트 컬럼','PART1')
    add('time__row__elapsed_h','raw__tx__cur__time__elapsed_h__wcur',
        '데이터시작 후 경과시간','timestamp','(ts-min)/3600','numeric','시간(h)','전체데이터','계산가능','전체기간위치->누수위험',False,'optional. 누수위험. used_in_ml=False','PART1','optional_leakage_risk')

    # categorical 5개 (기존 컬럼명 유지)
    _alias_map = {
        'from_bank':'raw__tx__cur__bank__from_enc__wcur',
        'to_bank':'raw__tx__cur__bank__to_enc__wcur',
        'payment_currency':'raw__tx__cur__ccy__paid_enc__wcur',
        'receiving_currency':'raw__tx__cur__ccy__recv_enc__wcur',
        'payment_format':'raw__tx__cur__payfmt__enc__wcur',
    }
    for raw_col, feat_col in CAT_FEATURE_MAP.items():
        j_alias = _alias_map.get(raw_col, '')
        add(feat_col, j_alias, f'{raw_col} train-only code', raw_col, 'train-only label code','numeric/int','코드(순서없음)','현재거래','unseen->-1','train only fit',True,'기존 전처리파트 컬럼. code값에 순서 의미 없음','PART1')

    # pair feature (candidate, 기존 naming 규칙 동일)
    _pair_j = {
        'cat__currency_pair__code':'raw__tx__cur__ccy__pair_enc__wcur',
        'cat__bank_pair__code':'raw__tx__cur__bank__pair_enc__wcur',
    }
    for pair_col, (col_a, col_b) in PAIR_ENCODING_DEFS.items():
        add(pair_col, _pair_j.get(pair_col,''), f'{col_a}+{col_b} pair code',
            f'{col_a},{col_b}', 'colA||colB train-only code','numeric/int','코드(순서없음)','현재거래','unseen pair->-1','train only fit',False,
            'candidate. 성능비교후 used_in_ml=True로 활성화 가능','PART2','candidate')

    # history feature
    for feat in ML01_HISTORY_FEATURE_COLS:
        add(feat, '', f'history: {feat}', 'sender/receiver,amount,timestamp',
            'window past-only agg','numeric','건수/금액',feat.split('__')[-1],'window없->0','ts<current',False,
            'ml_exp01 전용. 기본 used_in_ml=False. ml_feature_columns.csv에서 True로 변경 시 ml_exp01 실험에서 활성화 가능','PART2')

    return pd.DataFrame(records)


feature_catalog_df = make_feature_catalog()
catalog_path = CONFIG['LOCAL_OUTPUT_DIR'] / 'feature_catalog.csv'
feature_catalog_df.to_csv(catalog_path, index=False, encoding='utf-8-sig')
print('saved:', catalog_path)
# feature_name = 실제 parquet 컬럼명 / official_feature_name = ML파트 공식 표시명
display(feature_catalog_df[['feature_name','official_feature_name',
                             'description','used_in_ml','selection_status']].head(25))


## 14. leakage check / parquet 저장 / schema 호환성 검증

P0 leakage 점검 → 이상 없으면 parquet 저장 → schema 호환성 검증 순서로 실행합니다.  
`ml_feature_columns.csv`의 `used_in_ml=True` 컬럼이 저장된 parquet에 모두 존재하는지 확인합니다.

In [ ]:
# ============================================================
# ml_feature_columns.csv  (A안: ml_exp00 전용 feature가 기본 used_in_ml=True)
# history(ml_exp01) / amount_received 계열 / pair는 False 기본값
# LR baseline은 이 CSV를 읽으므로 ml_exp00 parquet과 충돌 없음
# ============================================================

def infer_feature_group(col):
    if col.startswith('amount__') or col.startswith('amount_received__'): return 'amount'
    if col.startswith('time__'):     return 'time'
    if col.startswith('cat__'):      return 'categorical_label_code'
    if col.startswith('timehist__'): return 'history'
    if col.startswith('recency__'):  return 'recency'
    if col.startswith('flag__'):     return 'flag'
    return 'unknown'

def infer_leakage_risk(col):
    if 'elapsed' in col:             return '누수위험_optional'
    if any(col.startswith(p) for p in ['timehist__','recency__','flag__']): return 'low_if_ts_lt_current'
    if col.startswith('cat__'):       return 'low_if_train_only_mapping'
    if col.startswith('time__'):      return 'low_current_ts_only'
    return 'low_current_row_only'

# 카탈로그에 포함할 모든 컬럼 목록
_HIST_SET  = set(ML01_HISTORY_FEATURE_COLS)
_RECV_SET  = {'amount_received__current__value','amount_received__current__log1p','amount__paid_recv_ratio'}
_PAIR_SET  = set(PAIR_FEATURE_COLS)
_ML00_SET  = set(ML00_FEATURE_COLS)  # 기본 used_in_ml=True 대상

def make_ml_feature_columns():
    all_cols = sorted(
        set(c for cols in FEATURE_SET_REGISTRY.values() for c in cols)
        | set(PAIR_COLS_IN_PARQUET)
        | set(AMOUNT_RECV_CANDIDATE_COLS)
    )
    records = []
    for col in all_cols:
        # used_in_ml 결정 (A안: ml_exp00 base만 True)
        if col in _ML00_SET:
            used, excl = True, ''
        elif col in _HIST_SET:
            used, excl = False, 'ml_exp01 history: 기본 False. ml_exp01 parquet 사용 시 True로 변경'
        elif col in _RECV_SET:
            used, excl = False, 'add_candidate: 기본 False. parquet에 있으면 True로 변경 가능'
        elif col in _PAIR_SET:
            used, excl = False, 'candidate: 성능비교 후 True로 변경 (parquet에 포함됨)'
        elif 'elapsed' in col:
            used, excl = False, '시간전체위치 누수위험 optional'
        else:
            used, excl = False, '미분류'

        # 어느 실험에 속하는지
        in_exps = ';'.join(n for n,cs in FEATURE_SET_REGISTRY.items() if col in cs)

        records.append({
            'column_name':        col,
            'used_in_ml':         used,
            'target_experiment':  'ml_exp00' if col in _ML00_SET else
                                  ('ml_exp01' if col in _HIST_SET else 'candidate'),
            'used_in_experiments': in_exps,
            'feature_group':      infer_feature_group(col),
            'dtype':              safe_schema_get(df_ml01_base.schema, col, 'unknown'),
            'excluded_reason':    excl,
            'leakage_risk':       infer_leakage_risk(col),
            'selection_note':     'candidate (parquet 포함)' if col in _PAIR_SET else '',
        })

    # meta/제외 컬럼
    for col in ['tx_id','timestamp','split','label','sender_account','receiver_account',
                'from_bank','to_bank','payment_currency','receiving_currency','payment_format']:
        if col in df_ml01_base.columns:
            records.append({
                'column_name':col,'used_in_ml':False,'target_experiment':'excluded',
                'used_in_experiments':'','feature_group':'meta_or_raw',
                'dtype':safe_schema_get(df_ml01_base.schema,col),
                'excluded_reason':'model input 제외','leakage_risk':'excluded','selection_note':''
            })
    return pd.DataFrame(records)

ml_feature_columns_df = make_ml_feature_columns()
ml_feature_columns_path = CONFIG['LOCAL_OUTPUT_DIR'] / 'ml_feature_columns.csv'
ml_feature_columns_df.to_csv(ml_feature_columns_path, index=False, encoding='utf-8-sig')
print('saved:', ml_feature_columns_path)
_n_true = ml_feature_columns_df['used_in_ml'].sum()
_n_total = len(ml_feature_columns_df)
print(f'used_in_ml=True: {_n_true} / total: {_n_total}')
print(f'True 컬럼은 모두 ml_exp00 parquet에 존재해야 합니다.')
display(ml_feature_columns_df[['column_name','used_in_ml','target_experiment',
                                'feature_group','excluded_reason']].head(40))


## column_mapping_review.csv — 전처리파트 ↔ ML파트 컬럼 매핑 비교표

전처리파트 output의 실제 parquet 컬럼명과 ML파트 공식 피처 명세 표준 표시명을 대조합니다.  
 계열은 parquet 실제 컬럼명이 아닌 **ML파트 공식 표시명**입니다.  
`suggested_action` 기준: **keep** (기존 유지) / **add_candidate** (후보 검토) / **exclude** (모델 제외) / **alias_only** (이름은 유지, catalog에 alias만 기록)  
**최종 모델 입력은 `ml_feature_columns.csv`의 `used_in_ml=True` 기준으로만 결정됩니다.**

In [ ]:
# ============================================================
# column_mapping_review.csv  (전처리파트 ↔ ML파트 컬럼 매핑 비교표)
#
# 목적:
#   전처리파트 parquet 실제 컬럼명과
#   ML파트 공식 피처 명세 문서의 표준 표시명을 한 테이블에서 대조
#
# 컬럼 설명:
#   parquet_column_name    : 실제 parquet에 저장되는 컬럼명 (전처리파트 output, 변경 금지)
#   official_feature_name  : ML파트 공식 피처 명세 문서의 표준 표시명
#                            (raw__tx__cur__... 계열은 parquet 컬럼이 아니라 명세 문서명)
#   suggested_action       : keep / add_candidate / alias_only / exclude
#   mapped_to_parquet_column: alias_only 행에서 실제 매핑 대상 parquet 컬럼명 명시
# ============================================================

_rows = [
    # parquet_col, official_feature_name, meaning, source_col, calc, current_status, suggested_action, used_in_ml, reason, notes, mapped_to_parquet_col
    # ── 금액 ──
    ('amount__current__value','raw__tx__cur__amt_paid__val__wcur','현재 거래 지급 금액','amount','identity','exists_in_output','keep',True,'기존 컬럼 정상','공식 표시명은 alias만 catalog 기록',''),
    ('amount__current__log1p','raw__tx__cur__amt__log_paid__wcur','지급 금액 log1p','amount','log1p','exists_in_output','keep',True,'기존 컬럼 정상','공식 표시명은 official_feature_name에만 기록',''),
    ('amount_received__current__value','raw__tx__cur__amt_recv__val__wcur','수취 금액','amount_received','identity','add_if_source_exists','add_candidate',False,'clean_base에 amount_received 있을 때만 생성. 기본 used_in_ml=False','parquet에 포함 시 used_in_ml=True로 변경 가능',''),
    ('amount_received__current__log1p','raw__tx__cur__amt__log_recv__wcur','수취 금액 log1p','amount_received','log1p','add_if_source_exists','add_candidate',False,'clean_base에 amount_received 있을 때만 생성. 기본 used_in_ml=False','',''),
    ('amount__paid_recv_ratio','raw__tx__cur__amt__paid_recv_ratio__wcur','지급/수취 금액 비율','amount,amount_received','paid/received','add_if_source_exists','add_candidate',False,'통화전환/금액불일치 포착. amount_received 필요. 기본 used_in_ml=False','received=0/null->null',''),
    # ── 시간 ──
    ('time__row__hour','raw__tx__cur__time__hour__wcur','거래 시간(0-23)','timestamp','dt.hour()','exists_in_output','keep',True,'기존 컬럼 정상','공식 표시명은 official_feature_name에만 기록',''),
    ('time__row__dayofweek','raw__tx__cur__time__dow__wcur','거래 요일(0=월~6=일)','timestamp','dt.weekday()-1','exists_in_output','keep',True,'기존 컬럼 정상. [QA] 0=월~6=일로 수정됨','공식 표시명은 official_feature_name에만 기록',''),
    ('time__row__is_weekend','raw__tx__cur__time__is_weekend__wcur','주말 여부(1=주말)','timestamp','(weekday-1)>=5->1','exists_in_output','keep',True,'기존 컬럼 정상. [QA] is_weekend 조건 수정됨','공식 표시명은 official_feature_name에만 기록',''),
    ('time__row__elapsed_h','raw__tx__cur__time__elapsed_h__wcur','데이터 시작 후 경과시간','timestamp','(ts-min)/3600','not_generated','add_candidate',False,'누수위험. optional. used_in_ml=False 유지','분석 목적 parquet 보존 시만 생성 가능',''),
    # ── categorical 5개 ──
    ('cat__from_bank__code','raw__tx__cur__bank__from_enc__wcur','송금 은행 code','from_bank','train-only label code','exists_in_output','keep',True,'기존 컬럼 정상. 코드값 순서의미없음','공식 표시명은 official_feature_name에만 기록',''),
    ('cat__to_bank__code','raw__tx__cur__bank__to_enc__wcur','수취 은행 code','to_bank','train-only label code','exists_in_output','keep',True,'기존 컬럼 정상','공식 표시명은 official_feature_name에만 기록',''),
    ('cat__payment_currency__code','raw__tx__cur__ccy__paid_enc__wcur','지급 통화 code','payment_currency','train-only label code','exists_in_output','keep',True,'기존 컬럼 정상','공식 표시명은 official_feature_name에만 기록',''),
    ('cat__receiving_currency__code','raw__tx__cur__ccy__recv_enc__wcur','수취 통화 code','receiving_currency','train-only label code','exists_in_output','keep',True,'기존 컬럼 정상','공식 표시명은 official_feature_name에만 기록',''),
    ('cat__payment_format__code','raw__tx__cur__payfmt__enc__wcur','결제 방식 code','payment_format','train-only label code','exists_in_output','keep',True,'기존 컬럼 정상','공식 표시명은 official_feature_name에만 기록',''),
    # ── pair feature (오늘 회의 후보) ──
    ('cat__currency_pair__code','raw__tx__cur__ccy__pair_enc__wcur','지급+수취 통화 쌍 code','payment_currency,receiving_currency','pair train-only code','generated_as_candidate','add_candidate',False,'오늘 회의 후보. parquet에 포함. 성능비교 후 used_in_ml=True 활성화 가능','ml_feature_columns.csv에서 True로 변경',''),
    ('cat__bank_pair__code','raw__tx__cur__bank__pair_enc__wcur','송금+수취 은행 쌍 code','from_bank,to_bank','pair train-only code','generated_as_candidate','add_candidate',False,'오늘 회의 후보. parquet에 포함. 성능비교 후 used_in_ml=True 활성화 가능','ml_feature_columns.csv에서 True로 변경',''),
    # ── ML파트 공식 표시명이 기존 컬럼의 alias인 경우 (alias_only) ──
    ('(없음-alias)','raw__tx__cur__bank__from_enc__wcur','from_bank code (ML파트명)','from_bank','-','alias_of_existing','alias_only',False,'cat__from_bank__code와 동일 값. 명칭만 다름. 실제 사용 여부는 mapped parquet 컬럼 기준','실제 사용 여부는 mapped parquet 컬럼 기준','cat__from_bank__code'),
    ('(없음-alias)','raw__tx__cur__bank__to_enc__wcur','to_bank code (ML파트명)','to_bank','-','alias_of_existing','alias_only',False,'cat__to_bank__code와 동일 값. 실제 사용 여부는 mapped parquet 컬럼 기준','실제 사용 여부는 mapped parquet 컬럼 기준','cat__to_bank__code'),
    ('(없음-alias)','raw__tx__cur__ccy__paid_enc__wcur','payment_currency code (ML파트명)','payment_currency','-','alias_of_existing','alias_only',False,'cat__payment_currency__code와 동일 값. 실제 사용 여부는 mapped parquet 컬럼 기준','실제 사용 여부는 mapped parquet 컬럼 기준','cat__payment_currency__code'),
    ('(없음-alias)','raw__tx__cur__ccy__recv_enc__wcur','receiving_currency code (ML파트명)','receiving_currency','-','alias_of_existing','alias_only',False,'cat__receiving_currency__code와 동일 값. 실제 사용 여부는 mapped parquet 컬럼 기준','실제 사용 여부는 mapped parquet 컬럼 기준','cat__receiving_currency__code'),
    ('(없음-alias)','raw__tx__cur__payfmt__enc__wcur','payment_format code (ML파트명)','payment_format','-','alias_of_existing','alias_only',False,'cat__payment_format__code와 동일 값. 실제 사용 여부는 mapped parquet 컬럼 기준','실제 사용 여부는 mapped parquet 컬럼 기준','cat__payment_format__code'),
    # ── meta / 제외 컬럼 ──
    ('label','(없음)','세탁 여부 라벨','Is Laundering','rename','exists_meta','exclude',False,'model input 제외. label leakage','parquet에 유지, ML feature 자동 차단(ml_io)',''),
    ('timestamp','(없음)','거래 시간 원본','Timestamp','cast datetime','exists_meta','exclude',False,'model input 제외. meta','parquet에 유지, Xy 파일에서 제거',''),
    ('tx_id','(없음)','거래 ID','없음/row index','auto gen','exists_meta','exclude',False,'model input 제외. ID','후속 파트 meta로 보존',''),
    ('sender_account','(없음)','송신 계좌','Account','rename','exists_meta','exclude',False,'model input 제외. 계좌ID는 후속 파트 meta','',''),
    ('receiver_account','(없음)','수신 계좌','Account.1','rename','exists_meta','exclude',False,'model input 제외. 계좌ID는 후속 파트 meta','',''),
    ('split','(없음)','train/val/test 구분','없음','time-sorted split','exists_meta','exclude',False,'model input 제외. split 컬럼','ml_io 자동 차단',''),
    # ── ML-01 history ──
    ('timehist__snd__out__tx_count__count__w1h','th__snd__out__txcnt__cnt__w1h','sender 1h 이전 송금 건수','sender_account,timestamp','window count past-only','exists_in_ml01','keep',False,'ml_exp01 관리. ts<current 조건. 기본 used_in_ml=False','ml_exp01 실험 시 활성화 가능',''),
    ('timehist__snd__out__amount__sum__w1d','th__snd__out__amt__sum__w1d','sender 1d 이전 송금 금액합','sender_account,amount,timestamp','window sum past-only','exists_in_ml01','keep',False,'ml_exp01 관리. 기본 used_in_ml=False','ml_exp01 실험 시 활성화 가능',''),
    ('(ML-01 history 전체)','th__* 계열','sender/receiver 과거 집계','sender/receiver,amount,timestamp','window past-only agg','exists_in_ml01','keep',False,'ml_exp01 관리. 기본 used_in_ml=False. 상세는 feature_catalog 참조','ml_exp01 실험 시 활성화 가능',''),
]

_cols = ['parquet_column_name','official_feature_name','feature_meaning',
         'source_column','calculation_method','current_status',
         'suggested_action','used_in_ml','reason','notes',
         'mapped_to_parquet_column']

# parquet_column_name = 실제 컬럼, official_feature_name = ML파트 명세 표시명
comparison_df = pd.DataFrame(_rows, columns=_cols)
comp_path = CONFIG['LOCAL_OUTPUT_DIR'] / 'column_mapping_review.csv'
comparison_df.to_csv(comp_path, index=False, encoding='utf-8-sig')
print('saved:', comp_path)
_action_counts = comparison_df['suggested_action'].value_counts()
print(_action_counts.to_string())
assert len(comparison_df) == 30, f'column_mapping_review 행 수 이상: {len(comparison_df)} (expected 30)'
display(comparison_df)


## 15. 검증 및 저장 (leakage check → parquet → schema 검증)

순서: leakage check → baseline metrics template → parquet 저장 → schema 호환성 검증

In [ ]:
# ============================================================
# leakage check
# P0 실패 시 파일 저장 중단
# ============================================================

LEAKAGE_RESULTS = []

# leakage check 결과를 같은 형식으로 모읍니다.
def add_check_result(check_name, status, severity, details, n_failed_rows=0):
    LEAKAGE_RESULTS.append({
        "check_name": check_name,
        "status": status,
        "severity": severity,
        "details": details,
        "n_failed_rows": int(n_failed_rows),
    })

    mark = "PASS" if status == "PASS" else "FAIL"
    print("[{}] [{}] {} | failed={} | {}".format(
        mark, severity, check_name, n_failed_rows, details
    ))


def _get_split_ts(summary_pd, split_name, col):
    sub = summary_pd.loc[summary_pd["split"] == split_name, col]
    if sub.empty:
        raise ValueError("split '{}' 가 비어 있습니다.".format(split_name))
    return pd.to_datetime(sub.iloc[0])


def validate_time_split(df):
    s = make_split_summary(df).to_pandas()

    train_max = _get_split_ts(s, "train", "timestamp_max")
    val_min = _get_split_ts(s, "val", "timestamp_min")
    val_max = _get_split_ts(s, "val", "timestamp_max")
    test_min = _get_split_ts(s, "test", "timestamp_min")

    ok = train_max <= val_min and val_max <= test_min

    add_check_result(
        "validate_time_split",
        "PASS" if ok else "FAIL",
        "P0",
        "train_max={}, val_min={}, val_max={}, test_min={}".format(
            train_max, val_min, val_max, test_min
        ),
        0 if ok else df.height,
    )


def validate_no_timestamp_overlap_between_splits(df):
    overlap = (
        df.group_by("timestamp")
        .agg(pl.col("split").n_unique().alias("n_splits"))
        .filter(pl.col("n_splits") > 1)
    )

    add_check_result(
        "validate_no_timestamp_overlap_between_splits",
        "PASS" if overlap.height == 0 else "FAIL",
        "P0",
        "same timestamp must not appear in multiple splits",
        overlap.height,
    )


def validate_no_forbidden_ml_columns(feature_cols, dataset_name):
    exact_forbidden = {x.lower() for x in CONFIG["EXACT_FORBIDDEN_MODEL_COLUMNS"]}
    substring_forbidden = {x.lower() for x in CONFIG["SUBSTRING_FORBIDDEN_MODEL_TOKENS"]}

    failed = []

    for col in feature_cols:
        lower_col = col.lower()

        # 파생 시간 피처와 recency 피처는 허용
        # 원본 timestamp 컬럼 자체는 exact forbidden에서 차단
        if col.startswith("time__row__") or col.startswith("recency__"):
            continue

        if lower_col in exact_forbidden:
            failed.append(col)
            continue

        for token in substring_forbidden:
            if token in lower_col:
                failed.append(col)
                break

    failed = sorted(set(failed))

    add_check_result(
        "validate_no_forbidden_ml_columns__{}".format(dataset_name),
        "PASS" if len(failed) == 0 else "FAIL",
        "P0",
        "forbidden columns: {}".format(failed[:20]),
        len(failed),
    )


def validate_categorical_mapping_train_only():
    if "category_mapping_train_only_df" not in globals():
        add_check_result(
            "validate_categorical_mapping_train_only",
            "FAIL",
            "P0",
            "category_mapping_train_only_df is missing",
            1,
        )
        return

    if len(CATEGORICAL_RAW_COLS) == 0:
        add_check_result(
            "validate_categorical_mapping_train_only",
            "PASS",
            "P0",
            "no categorical raw columns selected",
            0,
        )
        return

    bad_scope = category_mapping_train_only_df[
        category_mapping_train_only_df["fit_scope"] != "train_only"
    ]

    duplicate_feature_category = category_mapping_train_only_df.duplicated(
        subset=["raw_column", "encoding_type", "feature_column", "category"],
        keep=False,
    ).sum()

    ok = len(bad_scope) == 0 and duplicate_feature_category == 0

    add_check_result(
        "validate_categorical_mapping_train_only",
        "PASS" if ok else "FAIL",
        "P0",
        "bad_fit_scope_rows={}, duplicate_mapping_rows={}".format(
            len(bad_scope), int(duplicate_feature_category)
        ),
        len(bad_scope) + int(duplicate_feature_category),
    )


def validate_categorical_unknown_handling(df_with_features):
    failed = []

    for raw_col in CATEGORICAL_RAW_COLS:
        train_values = CATEGORY_ENCODERS[raw_col]["train_values"]
        code_col = CAT_FEATURE_MAP[raw_col]

        for split_name in ["val", "test"]:
            unknown_rows = df_with_features.filter(
                (pl.col("split") == split_name)
                & (~_normalized_string_expr(raw_col).is_in(list(train_values)))
            )

            if unknown_rows.height > 0:
                wrong_code = unknown_rows.filter(pl.col(code_col) != -1).height
                if wrong_code > 0:
                    failed.append({
                        "raw_column": raw_col,
                        "split": split_name,
                        "issue": "unseen_category_code_not_minus_one",
                        "n_failed_rows": wrong_code,
                    })

    add_check_result(
        "validate_categorical_unknown_handling",
        "PASS" if len(failed) == 0 else "FAIL",
        "P0",
        "first_failures={}".format(failed[:5]),
        sum(x["n_failed_rows"] for x in failed) if failed else 0,
    )


# 샘플 거래를 직접 재계산해서 history feature 누수를 확인합니다.
def direct_recompute_history_for_row(
    df,
    tx_id,
    entity_col,
    direction_label,
    entity_scope_label,
    window_name,
    window_str,
):
    row = df.filter(pl.col("tx_id") == tx_id)

    if row.height != 1:
        raise ValueError("tx_id row count must be 1: {}".format(tx_id))

    current_ts = row["timestamp"][0]
    entity_value = row[entity_col][0]
    lower_ts = current_ts - pd.Timedelta(window_str)

    past = df.filter(
        (pl.col(entity_col) == entity_value)
        & (pl.col("timestamp") >= lower_ts)
        & (pl.col("timestamp") < current_ts)
    )

    count_val = past.height
    sum_val = past["amount"].sum() if count_val > 0 else 0.0
    mean_val = sum_val / count_val if count_val > 0 else 0.0

    return {
        "timehist__{}__{}__tx_count__count__{}".format(
            entity_scope_label, direction_label, window_name
        ): count_val,
        "timehist__{}__{}__amount__sum__{}".format(
            entity_scope_label, direction_label, window_name
        ): float(sum_val),
        "timehist__{}__{}__amount__mean__{}".format(
            entity_scope_label, direction_label, window_name
        ): float(mean_val),
    }


# history feature가 현재/미래 거래를 참조하지 않는지 검증합니다.
def validate_no_current_or_future_history(df_with_features):
    n = df_with_features.height
    validation_mode = CONFIG.get("VALIDATION_MODE", "sample")

    if validation_mode not in ["sample", "full"]:
        raise ValueError("VALIDATION_MODE는 'sample' 또는 'full'만 허용합니다.")

    if validation_mode == "full":
        sample_df = df_with_features
        validation_rows = n
    else:
        validation_rows = min(CONFIG["MAX_VALIDATION_ROWS"], n)
        sample_df = df_with_features.sample(
            n=validation_rows,
            seed=CONFIG["RANDOM_SEED"],
            shuffle=True,
        )

    sample_tx_ids = sample_df["tx_id"].to_list()
    failed = []

    checks = []
    for w_name, w_str in windows.items():
        checks.append(("sender_account", "out", "sender", w_name, w_str))
        checks.append(("receiver_account", "in", "receiver", w_name, w_str))

    for tx_id in sample_tx_ids:
        for entity_col, direction_label, entity_scope_label, w_name, w_str in checks:
            recomputed = direct_recompute_history_for_row(
                df=df_with_features,
                tx_id=tx_id,
                entity_col=entity_col,
                direction_label=direction_label,
                entity_scope_label=entity_scope_label,
                window_name=w_name,
                window_str=w_str,
            )

            row = df_with_features.filter(pl.col("tx_id") == tx_id)

            for col_name, expected_val in recomputed.items():
                actual_val = row[col_name][0]

                if "mean" in col_name or "sum" in col_name:
                    # 금액 합계/평균은 float 저장 방식 때문에 아주 작은 오차가 생길 수 있음
                    ok = np.isclose(
                        float(actual_val),
                        float(expected_val),
                        rtol=1e-6,
                        atol=1e-4,
                    )
                else:
                    ok = int(actual_val) == int(expected_val)

                if not ok:
                    failed.append({
                        "tx_id": tx_id,
                        "feature": col_name,
                        "actual": actual_val,
                        "expected": expected_val,
                        "diff": float(actual_val) - float(expected_val),
                        "entity_col": entity_col,
                        "direction": direction_label,
                        "window": w_name,
                    })

                    if len(failed) >= 20:
                        break

            if len(failed) >= 20:
                break

        if len(failed) >= 20:
            break

    if len(failed) > 0:
        fail_path = CONFIG["LOCAL_OUTPUT_DIR"] / "leakage_history_failure_examples.csv"
        pd.DataFrame(failed).to_csv(fail_path, index=False, encoding="utf-8-sig")
    else:
        fail_path = None

    add_check_result(
        "validate_no_current_or_future_history",
        "PASS" if len(failed) == 0 else "FAIL",
        "P0",
        "validation_mode={}, validation_rows={}, total_rows={}, first_failures={}, failure_examples_path={}".format(
            validation_mode,
            validation_rows,
            n,
            failed[:5],
            fail_path,
        ),
        len(failed),
    )


def validate_first_sender_history_zero(df_with_features):
    first_col = "flag__sender__out__is_first_tx"

    if first_col not in df_with_features.columns:
        add_check_result(
            "validate_first_sender_history_zero",
            "FAIL",
            "P0",
            "missing column: {}".format(first_col),
            1,
        )
        return

    target_cols = [c for c in df_with_features.columns if c.startswith("timehist__sender__out__")]
    first_rows = df_with_features.filter(pl.col(first_col) == 1)

    failed_count = 0
    for c in target_cols:
        failed_count += first_rows.filter(pl.col(c).fill_null(0) != 0).height

    add_check_result(
        "validate_first_sender_history_zero",
        "PASS" if failed_count == 0 else "FAIL",
        "P0",
        "first sender rows should have zero sender history",
        failed_count,
    )


def validate_first_receiver_history_zero(df_with_features):
    first_col = "flag__receiver__in__is_first_tx"

    if first_col not in df_with_features.columns:
        add_check_result(
            "validate_first_receiver_history_zero",
            "FAIL",
            "P0",
            "missing column: {}".format(first_col),
            1,
        )
        return

    target_cols = [c for c in df_with_features.columns if c.startswith("timehist__receiver__in__")]
    first_rows = df_with_features.filter(pl.col(first_col) == 1)

    failed_count = 0
    for c in target_cols:
        failed_count += first_rows.filter(pl.col(c).fill_null(0) != 0).height

    add_check_result(
        "validate_first_receiver_history_zero",
        "PASS" if failed_count == 0 else "FAIL",
        "P0",
        "first receiver rows should have zero receiver history",
        failed_count,
    )


# leakage 검증 범위가 sample인지 full인지 리포트에 남깁니다.
def validate_history_validation_scope(df_with_features):
    n = df_with_features.height
    validation_mode = CONFIG.get("VALIDATION_MODE", "sample")

    if validation_mode == "full":
        validation_rows = n
        details = "validation_mode=full, validation_rows={}, total_rows={}".format(
            validation_rows,
            n,
        )
        severity = "INFO"
    elif validation_mode == "sample":
        validation_rows = min(CONFIG["MAX_VALIDATION_ROWS"], n)
        details = "validation_mode=sample, validation_rows={}, total_rows={}, MAX_VALIDATION_ROWS={}".format(
            validation_rows,
            n,
            CONFIG["MAX_VALIDATION_ROWS"],
        )
        severity = "P1"
    else:
        add_check_result(
            "validate_history_validation_scope",
            "FAIL",
            "P0",
            "invalid VALIDATION_MODE={}".format(validation_mode),
            1,
        )
        return

    add_check_result(
        "validate_history_validation_scope",
        "PASS",
        severity,
        details,
        0,
    )


def run_all_ml_validations():
    global LEAKAGE_RESULTS
    LEAKAGE_RESULTS = []

    _required_runtime_vars = [
        "df",
        "df_ml00_base",
        "df_ml01_base",
        "FEATURE_SET_REGISTRY",
        "category_mapping_train_only_df",
    ]
    _missing = [v for v in _required_runtime_vars if v not in globals()]
    if _missing:
        raise RuntimeError(
            "[RUN ORDER ERROR] 앞 셀이 실행되지 않아 필수 변수가 없습니다: "
            + ", ".join(_missing)
            + "\n런타임 재시작 후 전체 실행하거나, Cell 18~24를 먼저 실행한 뒤 leakage check를 실행하세요."
        )

    validate_history_validation_scope(df_ml01_base)

    validate_time_split(df)
    validate_no_timestamp_overlap_between_splits(df)

    for dataset_name, feature_cols in FEATURE_SET_REGISTRY.items():
        validate_no_forbidden_ml_columns(feature_cols, dataset_name)

    validate_categorical_mapping_train_only()
    validate_categorical_unknown_handling(df_ml00_base)

    validate_no_current_or_future_history(df_ml01_base)
    validate_first_sender_history_zero(df_ml01_base)
    validate_first_receiver_history_zero(df_ml01_base)

    leakage_df = pd.DataFrame(LEAKAGE_RESULTS)
    leakage_path = CONFIG["LOCAL_OUTPUT_DIR"] / "leakage_check.csv"
    leakage_df.to_csv(leakage_path, index=False, encoding="utf-8-sig")
    print("saved:", leakage_path)

    p0_fail = leakage_df[
        (leakage_df["severity"] == "P0")
        & (leakage_df["status"] != "PASS")
    ]

    if len(p0_fail) > 0:
        display(p0_fail)
        raise AssertionError("P0 leakage check failed. 파일 저장을 중단합니다.")

    return leakage_df


leakage_check = run_all_ml_validations()
leakage_check


In [ ]:
# ============================================================
# baseline metrics template
# ============================================================

# 추후 baseline metric을 채울 수 있도록 템플릿 행을 만듭니다.
def metric_row(experiment_name, split_name, data, notes):
    part = data.filter(pl.col("split") == split_name)

    return {
        "experiment_name": experiment_name,
        "split": split_name,
        "model_name": "",
        "n_rows": int(part.height),
        "positive_ratio": float(part["label"].mean()) if part.height > 0 else np.nan,
        "roc_auc": "",
        "pr_auc": "",
        "f1": "",
        "precision": "",
        "recall": "",
        "threshold": "",
        "notes": notes,
    }

metric_specs = [
    ("ml_exp00", ml_exp00, "ML파트 요청 기준: baseline current-row features with train-only label/code categorical encoding"),
    ("ml_exp00_no_time", ml_exp00_no_time, "ML파트 요청 기준: baseline without time features; train-only label/code categorical encoding"),
    ("ml_exp01", ml_exp01, "ML파트 요청 기준: ML-00 + Stage 0 history; train-only label/code categorical encoding"),
]

baseline_metrics_template = pd.DataFrame([
    metric_row(experiment_name, split_name, data, notes)
    for experiment_name, data, notes in metric_specs
    for split_name in ["train", "val", "test"]
])

metrics_template_path = CONFIG["LOCAL_OUTPUT_DIR"] / "ml_baseline_metrics_template.csv"
baseline_metrics_template.to_csv(metrics_template_path, index=False, encoding="utf-8-sig")

print("saved:", metrics_template_path)
baseline_metrics_template


In [ ]:
# ============================================================
# parquet 저장 — 단일 파일 전달 기본
#
# ML파트 피드백 반영:
#   - train/val/test를 별도 파일로 나누지 않음
#   - 하나의 parquet 안에 split 컬럼을 보존
#   - ML파트 코드는 split 컬럼을 읽어 train/val/test를 분리하면 됨
#
# 필요 시 아래 옵션을 True로 바꾸면 구버전 split 파일도 추가 저장 가능
# ============================================================

SAVE_SPLIT_PARQUETS = False   # 기본: False. train/val/test 분리 파일 생성 안 함
SAVE_XY_VARIANTS    = False   # 기본: False. Xy 중복 파일 생성 안 함


def save_parquet(df, filename):
    path = CONFIG['DRIVE_OUTPUT_DIR'] / filename
    df.write_parquet(path)
    print('saved:', path, '| rows=', df.height, '| cols=', len(df.columns))
    return path

# 기본 전달 파일: 실험별 단일 parquet 1개씩만 저장
DATASET_OBJECTS = {
    'ml_exp00': ml_exp00,
    'ml_exp00_no_time': ml_exp00_no_time,
    'ml_exp01': ml_exp01,
}

# 선택 옵션: 기존 Xy 파일도 필요할 때만 추가
if SAVE_XY_VARIANTS:
    DATASET_OBJECTS.update({
        'ml_exp00_Xy': ml_exp00_Xy,
        'ml_exp00_no_time_Xy': ml_exp00_no_time_Xy,
        'ml_exp01_Xy': ml_exp01_Xy,
    })

# 단일 parquet 저장 + split 컬럼 검증
for dataset_name, dataset_df in DATASET_OBJECTS.items():
    if 'split' not in dataset_df.columns:
        raise ValueError(f'[{dataset_name}] split 컬럼이 없습니다. 단일 파일 전달 불가')
    vals = set(dataset_df['split'].unique().to_list())
    if vals != {'train', 'val', 'test'}:
        raise ValueError(f'[{dataset_name}] split 값 이상: {sorted(vals)}')
    save_parquet(dataset_df, f'{dataset_name}.parquet')

# 구버전 호환용 split별 파일은 옵션이 켜진 경우에만 저장
if SAVE_SPLIT_PARQUETS:
    for dataset_name, dataset_df in DATASET_OBJECTS.items():
        for split_name in ['train', 'val', 'test']:
            save_parquet(
                dataset_df.filter(pl.col('split') == split_name),
                f'{dataset_name}_{split_name}.parquet',
            )
    print('[INFO] SAVE_SPLIT_PARQUETS=True → 구버전 train/val/test 분리 파일도 저장했습니다.')
else:
    print('[INFO] 단일 파일 전달: train/val/test 분리 parquet는 생성하지 않습니다.')

# 어떤 parquet을 전달해야 하는지 파일 목록 저장
file_list_df = pd.DataFrame([
    {
        'file_name': f'{name}.parquet',
        'rows': int(data.height),
        'cols': int(len(data.columns)),
        'has_split_column': 'split' in data.columns,
        'split_values': ','.join(sorted(map(str, data['split'].unique().to_list()))),
        'note': 'single parquet; split 컬럼으로 train/val/test 구분',
    }
    for name, data in DATASET_OBJECTS.items()
])
file_list_path = CONFIG['LOCAL_OUTPUT_DIR'] / '03_ml_feature_process_v2_file_list.csv'
file_list_df.to_csv(file_list_path, index=False, encoding='utf-8-sig')
print('saved:', file_list_path)
display(file_list_df)

record_memory('parquet_saved_single_file_output', ml_exp01)


## schema 호환성 검증

`ml_feature_columns.csv`의 `used_in_ml=True` 컬럼이 저장된 parquet에 모두 존재하는지 확인합니다.  
`ml_io.load_split` smoke test도 함께 실행합니다.

In [ ]:
# ============================================================
# schema 호환성 검증 — 단일 파일 전달 기준
# - split별 parquet가 아니라 단일 parquet 안의 split 컬럼을 검증
# - pyarrow로 schema/dtype 직접 확인
# - label 분포는 split별 전체 기준으로 확인
# ============================================================

import pyarrow.parquet as _pq
import numpy as np

_feat_csv_path = CONFIG['LOCAL_OUTPUT_DIR'] / 'ml_feature_columns.csv'
_ml_cols_df = pd.read_csv(_feat_csv_path, encoding='utf-8-sig')
_true_cols = _ml_cols_df.loc[_ml_cols_df['used_in_ml'] == True, 'column_name'].tolist()
print(f'used_in_ml=True 컬럼: {len(_true_cols)}개')

_EXPERIMENT_CHECKS = [
    ('ml_exp00',         ML00_FEATURE_COLS),
    ('ml_exp00_no_time', ML00_NO_TIME_FEATURE_COLS),
    ('ml_exp01',         ML01_FEATURE_COLS),
]
_compat_errors = []

for exp_name, feat_list in _EXPERIMENT_CHECKS:
    _pq_path = CONFIG['DRIVE_OUTPUT_DIR'] / f'{exp_name}.parquet'
    if not _pq_path.exists():
        _compat_errors.append(f'[{exp_name}] 단일 parquet 없음: {_pq_path}')
        continue
    try:
        _pf = _pq.ParquetFile(_pq_path)
        _schema = _pf.schema_arrow
        _schema_names = set(_schema.names)

        _required = set(feat_list) | {'split', 'label'}
        _missing = [c for c in _required if c not in _schema_names]
        if _missing:
            _compat_errors.append(f'[{exp_name}] 누락 컬럼 {len(_missing)}개: {_missing[:10]}')
            continue

        _non_num = []
        for col in feat_list:
            _idx = _schema.get_field_index(col)
            if _idx >= 0:
                _t = str(_schema.field(_idx).type)
                if not any(k in _t for k in ['int', 'float', 'double', 'uint', 'decimal']):
                    _non_num.append(f'{col}:{_t}')
        if _non_num:
            _compat_errors.append(f'[{exp_name}] non-numeric feature: {_non_num[:5]}')
        else:
            print(f'[OK] {exp_name}.parquet: schema+dtype 확인')

        _sdf = pd.read_parquet(_pq_path, columns=['split', 'label'])
        _split_vals = set(_sdf['split'].dropna().astype(str).unique().tolist())
        if _split_vals != {'train', 'val', 'test'}:
            _compat_errors.append(f'[{exp_name}] split 값 이상: {sorted(_split_vals)}')
        if _sdf['split'].isna().any():
            _compat_errors.append(f'[{exp_name}] split null 존재')

        print(f'  [Label 분포] {exp_name}')
        for split_name in ['train', 'val', 'test']:
            _part = _sdf[_sdf['split'].astype(str) == split_name]
            _n = len(_part)
            _pos = int((_part['label'] == 1).sum()) if _n else 0
            print(f'    {split_name}: n={_n:,} pos={_pos:,} '
                  f'pos_ratio={(_pos/_n if _n else 0):.5f} both_classes={0 < _pos < _n}')
    except Exception as _e:
        _compat_errors.append(f'[{exp_name}] schema/label 검증 실패: {_e}')

# used_in_ml=True vs ml_exp00 단일 parquet 교차 확인
_exp00_full = CONFIG['DRIVE_OUTPUT_DIR'] / 'ml_exp00.parquet'
if _exp00_full.exists():
    _exp00_cols = set(_pq.ParquetFile(_exp00_full).schema_arrow.names)
    _not_in = [c for c in _true_cols if c not in _exp00_cols]
    if _not_in:
        _compat_errors.append(f'[used_in_ml=True vs ml_exp00] 컬럼 없음: {_not_in}')
    else:
        print(f'[OK] used_in_ml=True {len(_true_cols)}개 컬럼 ml_exp00.parquet에 모두 존재')

# ml_io smoke test: 단일 parquet에서도 feature+label 로딩 가능 여부 확인
try:
    from ml_io import load_feature_columns as _lfc3, load_split as _ls3
    _smoke_feat = _lfc3(_feat_csv_path)
    if _exp00_full.exists():
        try:
            _Xl, _yl = _ls3(_exp00_full, _smoke_feat, label_col='label',
                            expected_split=None, sample_rows=50000, allow_nan=True)
            print(f'[OK] ml_io 단일 파일 smoke (50000행): X={_Xl.shape}, '
                  f'pos_ratio={_yl.mean():.5f} n_pos={int((_yl == 1).sum())}')
        except Exception as _e:
            _emsg = str(_e)
            if 'Both classes' in _emsg or 'both class' in _emsg.lower():
                print(f'[WARN] ml_io two-class smoke 실패: {_emsg}')
                print('       위 전체 label 분포를 참고하세요.')
            else:
                _compat_errors.append(f'[ml_io 단일 파일 smoke] 예상치 못한 실패: {_emsg}')
except ImportError:
    print('[SKIP] ml_io 없음 -> smoke test 건너뜀')

if _compat_errors:
    print('\n[COMPAT ERRORS]')
    for _e in _compat_errors:
        print(' -', _e)
    raise AssertionError(f'schema 오류 {len(_compat_errors)}건. 위 목록 확인.')
else:
    print('\n[COMPAT ALL OK] 단일 파일 parquet schema + dtype + split + used_in_ml=True 일치 확인')


## Optional ML 연동 smoke test: LR Baseline (ml_io + ml_metrics)

validation F1-optimal threshold -> test에 그대로 적용. Accuracy 단독 평가 금지.

In [ ]:
# ============================================================
# Optional ML 연동 smoke test: LR Baseline (ml_io + ml_metrics)
#
# 목적: 최종 성능 판단이 아니라, 전처리 산출물이 ML파트 코드와 연결되는지 확인
# 기본값 False: 전처리 전달본을 가볍게 유지
# 필요 시 True로 바꾸고 실행하면 lr_baseline_result.json을 저장함
# ============================================================

RUN_OPTIONAL_LR_BASELINE = False

import sys, time, tracemalloc, json as _json
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

_lr_json_path = CONFIG['LOCAL_OUTPUT_DIR'] / 'lr_baseline_result.json'
if not RUN_OPTIONAL_LR_BASELINE:
    if _lr_json_path.exists():
        _lr_json_path.unlink()  # optional 결과가 stale로 zip에 들어가지 않도록 제거
    print('[SKIP] Optional LR Baseline은 기본 실행하지 않습니다.')
    print('       필요 시 RUN_OPTIONAL_LR_BASELINE=True로 바꾸고 이 셀을 실행하세요.')
else:
    # ── ml_io / ml_metrics import smoke test ──────────────────────────
    try:
        from ml_io import load_feature_columns as _lfc, load_split as _ls
        from ml_metrics import select_threshold_by_f1, evaluate_at_threshold
        _HAS_MOD = True
        print('[OK] ml_io, ml_metrics import smoke test 통과')
    except ImportError as _e:
        _HAS_MOD = False
        print(f'[WARN] ml_io/ml_metrics import 실패: {_e} -> sklearn fallback')

    # ── feature 컬럼 로드 ─────────────────────────────────────────────
    _feat_csv = CONFIG['LOCAL_OUTPUT_DIR'] / 'ml_feature_columns.csv'
    if not _feat_csv.exists():
        raise FileNotFoundError(f'ml_feature_columns.csv 없음: {_feat_csv}')

    if _HAS_MOD:
        LR_FEAT = _lfc(_feat_csv)
        print(f'ml_feature_columns.csv -> {len(LR_FEAT)}개 feature')
    else:
        _fc_df = pd.read_csv(_feat_csv, encoding='utf-8-sig')
        LR_FEAT = _fc_df.loc[_fc_df['used_in_ml'] == True, 'column_name'].tolist()
        print(f'[FALLBACK pandas] ml_feature_columns.csv -> {len(LR_FEAT)}개 feature')

    _missing_in_exp00 = [c for c in LR_FEAT if c not in ml_exp00.columns]
    if _missing_in_exp00:
        raise ValueError('[LR ERROR] used_in_ml=True 컬럼이 ml_exp00에 없습니다:\n'
                         + '\n'.join(f'  - {c}' for c in _missing_in_exp00))

    # 단일 parquet smoke: train/val/test 파일을 요구하지 않음
    if _HAS_MOD:
        _full_pq = CONFIG['DRIVE_OUTPUT_DIR'] / 'ml_exp00.parquet'
        if _full_pq.exists():
            try:
                _X_s, _y_s = _ls(_full_pq, LR_FEAT, label_col='label',
                                  expected_split=None, sample_rows=50000,
                                  allow_nan=True)
                print(f'[OK] ml_io 단일 파일 sanity (50000행): X={_X_s.shape}, '
                      f'pos_ratio={_y_s.mean():.5f}')
            except Exception as _e:
                _emsg = str(_e)
                if ('Both classes' in _emsg or
                        'both class' in _emsg.lower() or
                        'one class' in _emsg.lower()):
                    print(f'[WARN] ml_io sanity one-class sample: {_emsg}')
                else:
                    raise

    def _Xy(df_pl, feats, split_name):
        sub = df_pl.filter(pl.col('split') == split_name)
        return (sub.select(feats).to_pandas().fillna(0),
                sub.select('label').to_pandas().squeeze().astype(int))

    X_tr, y_tr = _Xy(ml_exp00, LR_FEAT, 'train')
    X_va, y_va = _Xy(ml_exp00, LR_FEAT, 'val')
    X_te, y_te = _Xy(ml_exp00, LR_FEAT, 'test')
    print(f'train {X_tr.shape}, val {X_va.shape}, test {X_te.shape}')

    sc = StandardScaler()
    X_tr_s = sc.fit_transform(X_tr)
    X_va_s, X_te_s = sc.transform(X_va), sc.transform(X_te)

    tracemalloc.start()
    t0 = time.time()
    lr = LogisticRegression(max_iter=1000, class_weight='balanced',
                            solver='lbfgs', random_state=CONFIG['RANDOM_SEED'])
    lr.fit(X_tr_s, y_tr)
    fit_t = time.time() - t0
    _cur_mem, _peak_mem = tracemalloc.get_traced_memory()
    tracemalloc.stop()

    va_p = lr.predict_proba(X_va_s)[:, 1]
    te_p = lr.predict_proba(X_te_s)[:, 1]

    if _HAS_MOD:
        best_thr = select_threshold_by_f1(y_va, va_p)['threshold']
        va_m = evaluate_at_threshold(y_va, va_p, best_thr)['summary']
        te_m = evaluate_at_threshold(y_te, te_p, best_thr)['summary']
    else:
        from sklearn.metrics import (precision_recall_curve, f1_score,
            recall_score, precision_score, average_precision_score, confusion_matrix)
        _pr, _re, _th = precision_recall_curve(y_va, va_p)
        _f1s = 2 * _pr[:-1] * _re[:-1] / (_pr[:-1] + _re[:-1] + 1e-9)
        best_thr = float(_th[_f1s.argmax()])
        def _ev(y, p, t):
            _pd = (p >= t).astype(int)
            _tn, _fp, _fn, _tp = confusion_matrix(y, _pd, labels=[0, 1]).ravel()
            return {'threshold': t, 'f1': f1_score(y, _pd, zero_division=0),
                    'recall': recall_score(y, _pd, zero_division=0),
                    'precision': precision_score(y, _pd, zero_division=0),
                    'average_precision': average_precision_score(y, p),
                    'tn': int(_tn), 'fp': int(_fp), 'fn': int(_fn), 'tp': int(_tp)}
        va_m, te_m = _ev(y_va, va_p, best_thr), _ev(y_te, te_p, best_thr)

    t2 = time.time()
    lr.predict_proba(X_te_s)
    inf_t = time.time() - t2

    print('\n===== Optional LR Baseline =====')
    print(f'n_features={len(LR_FEAT)}, threshold={best_thr:.4f} (val F1-opt)')
    print(f'[VAL]  F1={va_m["f1"]:.4f} Recall={va_m["recall"]:.4f} '
          f'Precision={va_m["precision"]:.4f} AP={va_m["average_precision"]:.4f}')
    print(f'[TEST] F1={te_m["f1"]:.4f} Recall={te_m["recall"]:.4f} '
          f'Precision={te_m["precision"]:.4f} AP={te_m["average_precision"]:.4f}')

    _res = {
        'model': 'LogisticRegression',
        'feature_set': 'ml_exp00',
        'n_features': len(LR_FEAT),
        'threshold': best_thr,
        'fit_time_sec': round(fit_t, 4),
        'inference_time_test_sec': round(inf_t, 6),
        'memory_usage_mb': round(_cur_mem / 1024 / 1024, 2),
        'peak_memory_mb': round(_peak_mem / 1024 / 1024, 2),
        'val': va_m,
        'test': te_m,
        'note': 'optional ML 연동 smoke test; not final performance judgment',
    }
    with open(_lr_json_path, 'w', encoding='utf-8') as _f:
        _json.dump(_res, _f, indent=2, ensure_ascii=False)
    print('saved:', _lr_json_path)


In [ ]:
# ============================================================
# 마지막 확인 — 단일 파일 전달 기준
# ============================================================

save_memory_profile()

expected_parquet_bases = ['ml_exp00', 'ml_exp00_no_time', 'ml_exp01']
expected_outputs = [f'{base}.parquet' for base in expected_parquet_bases]

expected_outputs += [
    # ── feature 관리 ──
    'ml_feature_columns.csv',
    'feature_catalog.csv',
    'column_mapping_review.csv',
    '03_ml_feature_process_v2_file_list.csv',
    # ── split / 검증 ──
    'split_summary.csv',
    'split_assignment.csv',
    'leakage_check.csv',
    # ── encoding ──
    'category_mapping_train_only.csv',
    'categorical_encoding_summary.csv',
    'categorical_excluded_columns.csv',
    'pair_encoding_summary.csv',
    # ── baseline template / profile ──
    'ml_baseline_metrics_template.csv',
    'memory_profile.csv',
]

check_records = []
for fname in expected_outputs:
    path = (CONFIG['DRIVE_OUTPUT_DIR'] if fname.endswith('.parquet') else CONFIG['LOCAL_OUTPUT_DIR']) / fname
    check_records.append({
        'file_name': fname,
        'exists': path.exists(),
        'path': str(path),
        'size_mb': path.stat().st_size / 1024 / 1024 if path.exists() else None,
    })

output_check = pd.DataFrame(check_records)
display(output_check)

missing = output_check[~output_check['exists']]
if len(missing) > 0:
    raise FileNotFoundError('생성되지 않은 파일이 있습니다: {}'.format(missing['file_name'].tolist()))

# 단일 parquet에 split이 들어있는지 최종 확인
import pyarrow.parquet as _pq_check
for base in expected_parquet_bases:
    _path = CONFIG['DRIVE_OUTPUT_DIR'] / f'{base}.parquet'
    _cols = set(_pq_check.ParquetFile(_path).schema_arrow.names)
    if 'split' not in _cols:
        raise ValueError(f'{base}.parquet에 split 컬럼이 없습니다.')
    _tmp = pd.read_parquet(_path, columns=['split'])
    _vals = set(_tmp['split'].dropna().astype(str).unique().tolist())
    if _vals != {'train', 'val', 'test'}:
        raise ValueError(f'{base}.parquet split 값 이상: {sorted(_vals)}')
    print(f'[OK] {base}.parquet: single file + split column 확인')

# Optional LR 결과가 있으면 key만 검증하고, 없으면 정상 SKIP
_lr_json_path = CONFIG['LOCAL_OUTPUT_DIR'] / 'lr_baseline_result.json'
if _lr_json_path.exists():
    import json as _jmod
    _lr_data = _jmod.loads(_lr_json_path.read_text(encoding='utf-8'))
    _required_keys = ['val', 'test', 'threshold']
    _metric_keys = ['f1', 'recall', 'precision', 'average_precision']
    _missing_top = [k for k in _required_keys if k not in _lr_data]
    _missing_val = [k for k in _metric_keys if k not in _lr_data.get('val', {})]
    _missing_test = [k for k in _metric_keys if k not in _lr_data.get('test', {})]
    if _missing_top or _missing_val or _missing_test:
        raise ValueError('lr_baseline_result.json key 불완전')
    print(f'[OK] optional lr_baseline_result.json: threshold={_lr_data["threshold"]:.4f}')
else:
    print('[SKIP] optional lr_baseline_result.json 없음 — 전처리 전달 기준에서는 필수 아님')

print('완료')
print('drive output:', CONFIG['DRIVE_OUTPUT_DIR'])
print('local output:', CONFIG['LOCAL_OUTPUT_DIR'])
print('\n최종 전달 parquet:')
for base in expected_parquet_bases:
    print('-', f'{base}.parquet', ': train/val/test가 split 컬럼으로 포함된 단일 파일')


In [ ]:
# ============================================================
# Download helper: ML lightweight reports
# - Local report CSV/JSON을 zip으로 묶음
# - parquet 데이터는 Drive output에 단일 파일로 저장됨
# ============================================================

from pathlib import Path
import shutil

src_dir = Path('/content/GraphAML_local_outputs/ml_features')
zip_base = Path('/content/ml_features_reports')
zip_path = Path(str(zip_base) + '.zip')

print('local report dir:', src_dir)
print('exists:', src_dir.exists())

_required_local = [
    'ml_feature_columns.csv',
    'feature_catalog.csv',
    'column_mapping_review.csv',
    '03_ml_feature_process_v2_file_list.csv',
    'pair_encoding_summary.csv',
    'category_mapping_train_only.csv',
    'split_summary.csv',
    'split_assignment.csv',
    'leakage_check.csv',
    'memory_profile.csv',
]
_missing_files = [f for f in _required_local if not (src_dir / f).exists()]
if _missing_files:
    raise ValueError(
        f'[DOWNLOAD BLOCKED] 필수 산출물이 없어 zip을 생성할 수 없습니다.\n'
        f'누락 파일: {_missing_files}\n'
        '위 셀들을 순서대로 실행한 뒤 다시 시도해주세요.'
    )
print('[OK] 필수 산출물 전부 확인됨:')
for _rf in _required_local:
    _sz = (src_dir / _rf).stat().st_size / 1024
    print(f'  {_rf} ({_sz:.1f} KB)')

# optional LR 결과가 있으면 함께 포함됨
if (src_dir / 'lr_baseline_result.json').exists():
    print('[INFO] optional lr_baseline_result.json도 zip에 포함됩니다.')

if not src_dir.exists():
    raise FileNotFoundError(f'Local report directory not found: {src_dir}')

print('\nfiles:')
for f in sorted(src_dir.glob('*')):
    print('-', f.name)

if zip_path.exists():
    zip_path.unlink()

created_zip = shutil.make_archive(str(zip_base), 'zip', root_dir=src_dir)
print('\ncreated zip:', created_zip)

drive_backup_dir = Path('/content/drive/MyDrive/GraphAML/data/processed/ml_features')
drive_backup_dir.mkdir(parents=True, exist_ok=True)
backup_zip = drive_backup_dir / zip_path.name
shutil.copy2(zip_path, backup_zip)
print('copied to Drive:', backup_zip)

try:
    from google.colab import files
    files.download(str(zip_path))
except Exception as e:
    print('\n자동 다운로드를 실행할 수 없습니다.')
    print('Colab 왼쪽 파일 탭에서 직접 다운로드하세요:', zip_path)
    print('또는 Drive 백업 위치에서 다운로드하세요:', backup_zip)
    print('error:', e)


## 03_ml_feature_process_v2 요약  
### 핵심 원칙

| 항목 | 내용 |
|------|------|
| parquet 컬럼명 | **기존 전처리파트 output 유지** (`amount__current__value`, `cat__from_bank__code` 등) |
| ML파트 공식 표시명 | `feature_catalog.csv`의 `official_feature_name` + `column_mapping_review.csv`에만 기록 |
| split | `split` 컬럼으로 train/val/test를 구분합니다. split은 한 번 확정 후 `split_assignment.csv`로 고정·재사용합니다. |
| 전달 파일 형태 | train/val/test 분리 파일이 아니라, **하나의 parquet 안에 split 컬럼을 포함**합니다. |
| categorical encoding | 5개 컬럼 train-only code encoding. val/test unseen → -1 |

### 최종 전달 parquet

| 파일 | 설명 |
|------|------|
| `ml_exp00.parquet` | 기본 ML-00 feature. train/val/test가 `split` 컬럼으로 함께 들어 있음 |
| `ml_exp00_no_time.parquet` | time feature 제외 비교용. train/val/test가 `split` 컬럼으로 함께 들어 있음 |
| `ml_exp01.parquet` | history feature 포함 후보. train/val/test가 `split` 컬럼으로 함께 들어 있음 |

### feature 사용 기준 (`ml_feature_columns.csv`)

`column_name`은 실제 parquet 컬럼명 기준입니다. `official_feature_name`은 ML파트 명세 문서의 표준 표시명이며 parquet에 저장되는 컬럼명과 다릅니다.

| feature 그룹 | `used_in_ml` 기본값 | 비고 |
|---|---|---|
| ml_exp00 기본 feature (금액 2 + cat 5 + 시간 3) | **True** | 기본 전달 기준 |
| pair feature (`cat__currency_pair__code` 등) | False | parquet에 포함. 성능 비교 후 True로 변경 가능 |
| `amount_received` 계열 | False | clean_base에 amount_received 있을 때만 생성됨 |
| ML-01 history feature | False | ml_exp01 parquet 사용 시 True로 변경 |

### Optional LR Baseline

- LR Baseline은 최종 성능 판단이 아니라 **ML파트 연동 확인용 smoke test**입니다.
- 기본 실행하지 않으며, 필요할 때 `RUN_OPTIONAL_LR_BASELINE=True`로 바꿔 실행할 수 있습니다.
- 최종 성능 판단, 모델 튜닝, threshold 정책 결정은 ML파트에서 진행합니다.

### ML파트가 조정할 수 있는 항목

- `ml_feature_columns.csv`에서 `used_in_ml` 컬럼을 True/False로 변경하면 모델 입력이 바뀝니다.
- ML파트 코드는 단일 parquet를 읽은 뒤 `split` 컬럼으로 train/val/test를 분리하면 됩니다.
- **time feature 최종 사용 여부**: `ml_exp00` vs `ml_exp00_no_time` 비교로 결정 가능.
- **pair feature 활성화**: `cat__currency_pair__code`, `cat__bank_pair__code` → `used_in_ml=True`로 변경.
- **amount_received 계열**: clean_base에 컬럼 있고 성능 향상 확인 시 활성화.
- ML-01 history feature 실험: `ml_exp01.parquet` + `used_in_ml=True` 변경.

### 산출물 위치

| 파일 | 위치 | 설명 |
|------|------|------|
| `ml_exp00.parquet` 등 | Drive/ml_features | 단일 parquet 학습 데이터 (`split` 컬럼 포함) |
| `ml_feature_columns.csv` | Local/ml_features | **feature 활성화 기준** |
| `column_mapping_review.csv` | Local/ml_features | 전처리파트 parquet 컬럼명 ↔ ML파트 공식 표시명 매핑 비교표 |
| `feature_catalog.csv` | Local/ml_features | 전체 feature 정의 |
| `03_ml_feature_process_v2_file_list.csv` | Local/ml_features | 단일 parquet 전달 파일 목록 |
| `category_mapping_train_only.csv` | Local/ml_features | train-only 범주형 매핑 |

> **GNN/PageRank/경량화/모델 튜닝/feature 최종 선택은 이번 범위에서 제외합니다.**
